# CU Research Intelligence & Global Benchmarking

**Updated workflow**
- Upload only one dataset (Excel or CSV) for the CU EDA / AI research-area analysis.
- The 20 World + 20 India benchmark CSVs are embedded inside this notebook and unpack automatically.
- No separate upload of the 40 benchmark files is required.
- The Gradio dashboard includes cohort comparisons, portfolio gaps, rank comparisons, 40-university heatmap, institution explorer, SDG analysis, and one-file live analysis.

Run the notebook cells from top to bottom in Google Colab. The first upload prompt is for the dataset being analysed; the benchmark datasets are already built in.


In [ ]:
import os
import re
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from IPython.display import display

# Consistent, colorful charts
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 200

OUTPUT_DIR = "/content/CU_EDA_Results"
PLOTS_DIR = os.path.join(OUTPUT_DIR, "plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

print("Upload ONE university research dataset (Excel or CSV) for EDA and AI prediction.")
uploaded = files.upload()

if not uploaded:
    raise ValueError("Please upload one Excel (.xlsx/.xls) or CSV file.")

file_name = next(iter(uploaded))
extension = os.path.splitext(file_name)[1].lower()

if extension in [".xlsx", ".xls"]:
    df = pd.read_excel(file_name)
elif extension == ".csv":
    try:
        df = pd.read_csv(file_name, encoding="utf-8-sig", low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(file_name, encoding="latin-1", low_memory=False)
else:
    raise ValueError("Unsupported file type. Please upload .xlsx, .xls, or .csv.")

df.columns = df.columns.astype(str).str.strip()

print("=" * 60)
print("CHANDIGARH UNIVERSITY - DATASET OVERVIEW")
print("=" * 60)

print("File:", file_name)
print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])
print("Memory usage (MB):",
      round(df.memory_usage(deep=True).sum() / (1024**2), 2))

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 10 records:")
display(df.head(10))

print("\nColumn types and non-null counts:")
df.info()

print("\nData types summary:")
display(df.dtypes.astype(str).value_counts().rename_axis(
    "Data Type"
).reset_index(name="Number of Columns"))

schema_report = pd.DataFrame({
    "Column": df.columns,
    "Data Type": [str(dtype) for dtype in df.dtypes],
    "Non-Null Count": df.notna().sum().values,
    "Missing Count": df.isna().sum().values,
    "Missing Percentage": (df.isna().mean().values * 100).round(2)
})

display(schema_report)
schema_report.to_csv(os.path.join(OUTPUT_DIR, "dataset_schema.csv"), index=False)


In [ ]:

print("Available columns:")
for i, col in enumerate(df.columns):
    print(f"{i}: {col}")

# Automatically suggest likely column names
def find_column(keywords):
    for col in df.columns:
        name = col.lower().strip()
        if any(k in name for k in keywords):
            return col
    return None

suggested_title = find_column([
    "publication title", "paper title",
    "document title", "article title", "title"
])

suggested_cluster = find_column([
    "cluster", "research area", "research domain",
    "topic", "category"
])

print("\nSuggested title column:", suggested_title)
print("Suggested cluster column:", suggested_cluster)

# CHANGE THESE TWO VALUES if suggestions are incorrect.
TITLE_COL = suggested_title
CLUSTER_COL = suggested_cluster

if TITLE_COL is None or CLUSTER_COL is None:
    raise ValueError(
        "Set TITLE_COL and CLUSTER_COL manually using the exact "
        "column names printed above."
    )

if TITLE_COL == CLUSTER_COL:
    raise ValueError("Title and cluster columns must be different.")

print("\nSelected title column:", TITLE_COL)
print("Selected cluster column:", CLUSTER_COL)

display(df[[TITLE_COL, CLUSTER_COL]].head(10))


In [ ]:

# Preserve the original dataframe; use a separate analysis copy
analysis_df = df.copy()

# Convert blank/whitespace-only cells to missing values
# only in the analysis copy
for col in analysis_df.columns:
    if (
        pd.api.types.is_object_dtype(analysis_df[col])
        or pd.api.types.is_string_dtype(analysis_df[col])
    ):
        analysis_df[col] = analysis_df[col].replace(
            r"^\s*$", np.nan, regex=True
        )

# Missing values
missing_report = pd.DataFrame({
    "Column": analysis_df.columns,
    "Missing Count": analysis_df.isna().sum().values,
    "Missing Percentage": (
        analysis_df.isna().mean().values * 100
    ).round(2)
}).sort_values("Missing Count", ascending=False)

print("MISSING VALUES REPORT")
display(missing_report)

# Exact duplicate rows across all columns
exact_duplicate_mask = analysis_df.duplicated(keep="first")
exact_duplicate_count = int(exact_duplicate_mask.sum())

print("\nExact duplicate rows:", exact_duplicate_count)

# Repeated titles, ignoring case and surrounding spaces
title_key = (
    analysis_df[TITLE_COL]
    .astype("string")
    .str.strip()
    .str.casefold()
)

valid_title_mask = title_key.notna() & title_key.ne("")

repeated_title_mask = (
    valid_title_mask
    & title_key.duplicated(keep=False)
)

repeated_title_rows = analysis_df.loc[
    repeated_title_mask
].copy()

print("Rows with a repeated non-empty title:",
      int(repeated_title_mask.sum()))

print("Unique titles appearing more than once:",
      int(title_key[repeated_title_mask].nunique()))

# Check whether repeated titles appear in multiple clusters
repeated_title_cluster_check = analysis_df.loc[
    valid_title_mask, [TITLE_COL, CLUSTER_COL]
].copy()

repeated_title_cluster_check["_title_key"] = title_key[valid_title_mask]

repeated_title_cluster_check["_cluster_key"] = (
    repeated_title_cluster_check[CLUSTER_COL]
    .astype("string")
    .str.strip()
    .str.casefold()
)

repeated_title_cluster_check = (
    repeated_title_cluster_check.groupby("_title_key")
    .agg(
        Number_of_Rows=(TITLE_COL, "size"),
        Number_of_Distinct_Clusters=("_cluster_key", "nunique"),
        Cluster_Values=(
            CLUSTER_COL,
            lambda x: " | ".join(
                sorted(set(x.dropna().astype(str)))
            )
        )
    )
    .reset_index()
)

repeated_title_cluster_check = (
    repeated_title_cluster_check[
        repeated_title_cluster_check["Number_of_Rows"] > 1
    ]
)

print("\nRepeated-title analysis:")
display(repeated_title_cluster_check.head(30))

missing_report.to_csv(
    os.path.join(OUTPUT_DIR, "missing_values_report.csv"),
    index=False
)

repeated_title_rows.to_csv(
    os.path.join(OUTPUT_DIR, "repeated_title_records.csv"),
    index=False
)

repeated_title_cluster_check.to_csv(
    os.path.join(OUTPUT_DIR, "repeated_title_cluster_check.csv"),
    index=False
)


In [ ]:

# Standardized keys are used for counting and comparison.
# Original labels remain available for reporting.

title_values = (
    analysis_df[TITLE_COL]
    .astype("string")
    .str.strip()
)

cluster_values = (
    analysis_df[CLUSTER_COL]
    .astype("string")
    .str.strip()
)

valid_titles = title_values.notna() & title_values.ne("")
valid_clusters = cluster_values.notna() & cluster_values.ne("")

title_keys = title_values.str.casefold()
cluster_keys = cluster_values.str.casefold()

total_rows = len(analysis_df)
unique_title_count = title_keys[valid_titles].nunique()
unique_cluster_count = cluster_keys[valid_clusters].nunique()

# Count cluster records without deleting repeated titles.
cluster_work = pd.DataFrame({
    "Cluster": cluster_values,
    "_cluster_key": cluster_keys,
    "Title": title_values,
    "_title_key": title_keys
})

# Blank cluster labels are reported separately as unassigned
cluster_work["Cluster"] = cluster_work["Cluster"].where(
    valid_clusters, "Missing / Unassigned"
)

cluster_summary = (
    cluster_work.groupby(
        ["_cluster_key", "Cluster"], dropna=False
    )
    .agg(
        Publication_Records=("Title", "size"),
        Non_Empty_Title_Records=("_title_key", lambda x: int(x.notna().sum())),
        Unique_Titles=("_title_key", "nunique")
    )
    .reset_index()
    .drop(columns="_cluster_key")
    .sort_values("Publication_Records", ascending=False)
)

# Percentages use all dataset records as the denominator.
# Therefore, the cluster percentages sum to 100% when each row
# has exactly one cluster value (including the missing category).
cluster_summary["Percentage_of_All_Records"] = (
    cluster_summary["Publication_Records"]
    / total_rows * 100
).round(2) if total_rows else 0

# Unique title count is a separate statistic, not the cluster count.
print("Total dataset rows:", total_rows)
print("Unique non-empty publication titles:", unique_title_count)
print("Unique non-empty clusters:", unique_cluster_count)
print("Rows with missing/blank titles:", int((~valid_titles).sum()))
print("Rows with missing/blank clusters:", int((~valid_clusters).sum()))

print("\nCluster-wise summary:")
display(cluster_summary)

cluster_summary.to_csv(
    os.path.join(OUTPUT_DIR, "cluster_wise_summary.csv"),
    index=False
)


In [ ]:

# GRAPH 1: Missing values by column
missing_plot = missing_report[
    missing_report["Missing Count"] > 0
].sort_values("Missing Percentage")

if not missing_plot.empty:
    plt.figure(figsize=(11, max(4, len(missing_plot) * 0.4)))
    sns.barplot(
        data=missing_plot,
        x="Missing Percentage",
        y="Column",
        hue="Column",
        palette="Reds_r",
        legend=False
    )
    plt.title("Missing Values by Column", fontsize=15, weight="bold")
    plt.xlabel("Missing values (%)")
    plt.ylabel("")
    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/01_missing_values.png", bbox_inches="tight")
    plt.show()
else:
    print("Graph 1: No missing values found.")

# GRAPH 2: Dataset completeness
complete = int(analysis_df.notna().all(axis=1).sum())
incomplete = total_rows - complete

plt.figure(figsize=(7, 5))
plt.bar(
    ["Complete rows", "Rows with missing values"],
    [complete, incomplete],
    color=["#2ECC71", "#E67E22"]
)
plt.title("Dataset Row Completeness", fontsize=14, weight="bold")
plt.ylabel("Number of rows")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/02_row_completeness.png", bbox_inches="tight")
plt.show()

# GRAPH 3: Exact duplicates vs non-duplicate rows
non_duplicate_rows = total_rows - exact_duplicate_count

plt.figure(figsize=(7, 5))
plt.bar(
    ["Non-duplicate rows", "Exact duplicate rows"],
    [non_duplicate_rows, exact_duplicate_count],
    color=["#3498DB", "#E74C3C"]
)
plt.title("Exact Duplicate Row Check", fontsize=14, weight="bold")
plt.ylabel("Number of rows")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/03_exact_duplicates.png", bbox_inches="tight")
plt.show()

# GRAPH 4: All clusters, sorted by record count
plot_data = cluster_summary.sort_values("Publication_Records")

plt.figure(figsize=(12, max(5, len(plot_data) * 0.4)))
sns.barplot(
    data=plot_data,
    x="Publication_Records",
    y="Cluster",
    hue="Cluster",
    palette="husl",
    legend=False
)
plt.title("Publication Records Across All Clusters",
          fontsize=15, weight="bold")
plt.xlabel("Number of records")
plt.ylabel("Cluster")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/04_all_clusters.png", bbox_inches="tight")
plt.show()

# GRAPH 5: Percentage share of all records
share_data = cluster_summary.sort_values(
    "Percentage_of_All_Records"
)

plt.figure(figsize=(12, max(5, len(share_data) * 0.4)))
sns.barplot(
    data=share_data,
    x="Percentage_of_All_Records",
    y="Cluster",
    hue="Cluster",
    palette="viridis",
    legend=False
)
plt.title("Percentage of Total Dataset Records by Cluster",
          fontsize=15, weight="bold")
plt.xlabel("Percentage of all records (%)")
plt.ylabel("Cluster")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/05_cluster_percentages.png", bbox_inches="tight")
plt.show()

# GRAPH 6: Repeated publication titles vs single-occurrence titles
title_frequency = title_keys[valid_titles].value_counts()

single_occurrence_titles = int((title_frequency == 1).sum())
repeated_unique_titles = int((title_frequency > 1).sum())

plt.figure(figsize=(8, 5))
plt.bar(
    ["Titles appearing once", "Titles appearing multiple times"],
    [single_occurrence_titles, repeated_unique_titles],
    color=["#9B59B6", "#F39C12"]
)
plt.title("Unique Title Frequency Check", fontsize=14, weight="bold")
plt.ylabel("Number of distinct titles")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/06_title_repetition.png", bbox_inches="tight")
plt.show()

print("All applicable EDA graphs generated.")


In [ ]:

# Create final summary table
final_metrics = pd.DataFrame({
    "Metric": [
        "Total rows",
        "Total columns",
        "Unique non-empty publication titles",
        "Unique non-empty clusters",
        "Rows with missing/blank titles",
        "Rows with missing/blank clusters",
        "Exact duplicate rows",
        "Distinct titles appearing multiple times",
        "Rows containing at least one missing value"
    ],
    "Value": [
        total_rows,
        analysis_df.shape[1],
        unique_title_count,
        unique_cluster_count,
        int((~valid_titles).sum()),
        int((~valid_clusters).sum()),
        exact_duplicate_count,
        repeated_unique_titles,
        int(analysis_df.isna().any(axis=1).sum())
    ]
})

excel_path = os.path.join(
    OUTPUT_DIR, "Chandigarh_University_EDA_Report.xlsx"
)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    final_metrics.to_excel(
        writer, sheet_name="Dataset Overview", index=False
    )
    schema_report.to_excel(
        writer, sheet_name="Column Schema", index=False
    )
    missing_report.to_excel(
        writer, sheet_name="Missing Values", index=False
    )
    cluster_summary.to_excel(
        writer, sheet_name="Cluster Summary", index=False
    )
    repeated_title_cluster_check.to_excel(
        writer, sheet_name="Repeated Titles", index=False
    )
    repeated_title_rows.to_excel(
        writer, sheet_name="Repeated Title Records", index=False
    )

# Save a clean, auditable analysis copy; original df remains unchanged.
analysis_df.to_csv(
    os.path.join(OUTPUT_DIR, "analysis_dataset.csv"),
    index=False
)

# Package all reports and graphs
zip_path = "/content/Chandigarh_University_EDA_Results.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk(OUTPUT_DIR):
        for filename in filenames:
            full_path = os.path.join(root, filename)
            arcname = os.path.relpath(full_path, OUTPUT_DIR)
            zf.write(full_path, arcname)

print("EDA completed.")
print("Excel report:", excel_path)
print("ZIP package:", zip_path)

display(final_metrics)

files.download(excel_path)
files.download(zip_path)


In [ ]:

# UNIVERSAL EDA - A: Detect optional columns

def find_matching_columns(columns, keywords):
    matches = []

    for col in columns:
        normalized = re.sub(r"[^a-z0-9]+", " ", str(col).lower()).strip()

        if any(
            re.search(r"\b" + re.escape(keyword) + r"\b", normalized)
            for keyword in keywords
        ):
            matches.append(col)

    return matches


COLUMN_PATTERNS = {
    "year": ["year", "publication year", "published year"],
    "date": ["publication date", "published date", "date"],
    "author": ["author", "authors", "researcher"],
    "keyword": ["keyword", "keywords", "index terms"],
    "abstract": ["abstract", "summary"],
    "journal": ["journal", "conference", "venue", "source title"],
    "doi": ["doi", "digital object identifier"],
    "citation": ["citation count", "citations", "cited by"],
    "department": ["department", "faculty", "school"],
    "institution": ["institution", "university", "organization"],
    "sdg": ["sdg", "sustainable development goal"],
    "subject": ["subject area", "research area", "field of study"]
}

detected_columns = {}

for category, keywords in COLUMN_PATTERNS.items():
    detected_columns[category] = find_matching_columns(
        analysis_df.columns, keywords
    )

print("OPTIONAL COLUMN DETECTION")
for category, cols in detected_columns.items():
    print(f"{category}: {cols if cols else 'Not available'}")

# Save detection report
detection_report = pd.DataFrame([
    {
        "Category": category,
        "Detected Columns": ", ".join(cols) if cols else "Not available"
    }
    for category, cols in detected_columns.items()
])

display(detection_report)


In [ ]:

# UNIVERSAL EDA - B: Publication trends by year

year_cols = detected_columns["year"]
date_cols = detected_columns["date"]

year_col = year_cols[0] if year_cols else None
date_col = date_cols[0] if date_cols else None

yearly_summary = None

if year_col:
    years = pd.to_numeric(analysis_df[year_col], errors="coerce")

elif date_col:
    years = pd.to_datetime(
        analysis_df[date_col], errors="coerce"
    ).dt.year

else:
    years = None

if years is not None:

    valid_years = years.between(1900, 2100)

    year_data = pd.DataFrame({
        "Year": years[valid_years].astype(int)
    })

    if CLUSTER_COL in analysis_df.columns:
        year_data["Cluster"] = (
            analysis_df.loc[valid_years, CLUSTER_COL]
            .astype("string")
            .fillna("Missing / Unassigned")
            .values
        )

    yearly_summary = (
        year_data.groupby("Year")
        .size()
        .reset_index(name="Publication Records")
        .sort_values("Year")
    )

    display(yearly_summary)

    plt.figure(figsize=(12, 5))
    sns.lineplot(
        data=yearly_summary,
        x="Year",
        y="Publication Records",
        marker="o",
        color="#2878B5"
    )

    plt.title("Publication Records by Year")
    plt.tight_layout()
    plt.savefig(
        os.path.join(PLOTS_DIR, "optional_yearly_trends.png"),
        bbox_inches="tight"
    )
    plt.show()

    if "Cluster" in year_data.columns:
        top_clusters = (
            year_data["Cluster"].value_counts().head(10).index
        )

        trend_data = year_data[
            year_data["Cluster"].isin(top_clusters)
        ]

        if not trend_data.empty:
            cluster_year_counts = (
                trend_data.groupby(["Year", "Cluster"])
                .size()
                .reset_index(name="Publication Records")
            )

            plt.figure(figsize=(13, 6))
            sns.lineplot(
                data=cluster_year_counts,
                x="Year",
                y="Publication Records",
                hue="Cluster",
                marker="o"
            )

            plt.title("Publication Trends by Cluster")
            plt.tight_layout()
            plt.savefig(
                os.path.join(PLOTS_DIR, "optional_cluster_year_trends.png"),
                bbox_inches="tight"
            )
            plt.show()

else:
    print("No usable year/date column found. Year analysis skipped.")


In [ ]:

# UNIVERSAL EDA - C: Analyze categorical / multi-value fields

def split_multi_value(value):
    if pd.isna(value):
        return []

    # Common separators used in exported publication datasets
    parts = re.split(r"\s*[;|]\s*|\s*,\s*", str(value))

    return [
        p.strip() for p in parts
        if p.strip()
    ]


categorical_categories = [
    "author", "keyword", "journal",
    "department", "institution", "sdg", "subject"
]

optional_category_tables = {}

for category in categorical_categories:

    cols = detected_columns[category]

    if not cols:
        print(f"{category}: no column found; skipped.")
        continue

    col = cols[0]

    values = (
        analysis_df[col]
        .dropna()
        .apply(split_multi_value)
        .explode()
        .dropna()
        .astype(str)
        .str.strip()
    )

    values = values[values.ne("")]

    if values.empty:
        print(f"{category}: no usable values; skipped.")
        continue

    result = (
        values.value_counts()
        .rename_axis(category.title())
        .reset_index(name="Record Occurrences")
    )

    optional_category_tables[category] = result

    print(f"\n{category.upper()} - most frequent values")
    display(result.head(20))

    result.to_csv(
        os.path.join(
            OUTPUT_DIR, f"optional_{category}_frequency.csv"
        ),
        index=False
    )

    chart_data = result.head(15).sort_values("Record Occurrences")

    plt.figure(figsize=(11, max(4, len(chart_data) * 0.35)))
    sns.barplot(
        data=chart_data,
        x="Record Occurrences",
        y=category.title(),
        hue=category.title(),
        palette="husl",
        legend=False
    )

    plt.title(f"Most Frequent {category.title()} Values")
    plt.tight_layout()

    plt.savefig(
        os.path.join(PLOTS_DIR, f"optional_{category}.png"),
        bbox_inches="tight"
    )
    plt.show()


In [ ]:

# UNIVERSAL EDA - D: Citation and identifier analysis

citation_cols = detected_columns["citation"]
doi_cols = detected_columns["doi"]
abstract_cols = detected_columns["abstract"]

if citation_cols:

    citation_col = citation_cols[0]

    citations = pd.to_numeric(
        analysis_df[citation_col], errors="coerce"
    )

    print("Citation statistics:")
    display(citations.describe().to_frame("Citation Count"))

    print("Missing or non-numeric citation values:",
          int(citations.isna().sum()))

    if citations.notna().any():

        plt.figure(figsize=(10, 5))
        sns.histplot(
            citations.dropna(),
            bins=30,
            color="#8E6BBE"
        )

        plt.title("Citation Count Distribution")
        plt.xlabel("Citation count")
        plt.tight_layout()

        plt.savefig(
            os.path.join(PLOTS_DIR, "optional_citation_distribution.png"),
            bbox_inches="tight"
        )
        plt.show()

else:
    print("Citation column not found. Citation analysis skipped.")

if doi_cols:

    doi_col = doi_cols[0]
    doi_values = (
        analysis_df[doi_col]
        .astype("string")
        .str.strip()
    )

    valid_doi = doi_values.notna() & doi_values.ne("")

    print("Rows with DOI:", int(valid_doi.sum()))
    print("Unique non-empty DOI values:",
          int(doi_values[valid_doi].nunique()))

    print("Repeated DOI values:",
          int(doi_values[valid_doi].duplicated().sum()))

else:
    print("DOI column not found. DOI analysis skipped.")

if abstract_cols:

    abstract_col = abstract_cols[0]
    abstract_values = (
        analysis_df[abstract_col]
        .astype("string")
        .str.strip()
    )

    valid_abstract = (
        abstract_values.notna() & abstract_values.ne("")
    )

    print("Rows with abstracts:", int(valid_abstract.sum()))
    print("Rows without abstracts:",
          int((~valid_abstract).sum()))

else:
    print("Abstract column not found. Abstract analysis skipped.")


In [ ]:

# UNIVERSAL EDA - E: Export optional results

optional_excel_path = os.path.join(
    OUTPUT_DIR, "CU_Optional_EDA_Results.xlsx"
)

with pd.ExcelWriter(optional_excel_path, engine="openpyxl") as writer:

    detection_report.to_excel(
        writer, sheet_name="Detected Columns", index=False
    )

    if yearly_summary is not None:
        yearly_summary.to_excel(
            writer, sheet_name="Yearly Trends", index=False
        )

    for category, table in optional_category_tables.items():

        # Excel sheet names must be <= 31 characters
        sheet_name = f"{category.title()} Frequency"[:31]

        table.to_excel(
            writer, sheet_name=sheet_name, index=False
        )

print("Optional EDA workbook created:", optional_excel_path)


In [ ]:

%pip install -q sentence-transformers pandas numpy matplotlib seaborn openpyxl


In [ ]:

research_areas = {
    "Medicine": [
        "clinical medicine, diseases, diagnosis, treatment, healthcare",
        "medical research, clinical trials, surgery, patient outcomes",
        "human health, medical conditions, therapies and clinical studies"
    ],
    "Physics and Astronomy": [
        "physics, quantum mechanics, optics, particles and physical laws",
        "astronomy, astrophysics, stars, galaxies and cosmology",
        "theoretical physics and experimental physics"
    ],
    "Engineering": [
        "engineering design, mechanical systems and industrial technology",
        "electrical, civil, robotics and control engineering",
        "engineering methods, manufacturing and technical applications"
    ],
    "Biochemistry, Genetics and Molecular Biology": [
        "DNA, genes, genetic variation and molecular biology",
        "proteins, enzymes, cells and biochemical mechanisms",
        "genomics, gene expression and biochemical pathways"
    ],
    "Computer Science": [
        "artificial intelligence, machine learning and deep learning",
        "computer algorithms, software systems and data science",
        "cybersecurity, networks, computer vision and natural language processing"
    ],
    "Materials Science": [
        "nanomaterials, polymers, metals and composite materials",
        "material characterization, properties and crystal structures",
        "advanced functional materials and materials engineering"
    ],
    "Chemistry": [
        "organic, inorganic and analytical chemistry",
        "chemical reactions, molecular structures and synthesis",
        "catalysis, compounds and chemical laboratory research"
    ],
    "Social Sciences": [
        "sociology, society, social behavior and social inequality",
        "education, public policy, governance and communities",
        "social science research on people and institutions"
    ],
    "Mathematics": [
        "mathematical analysis, algebra and differential equations",
        "statistics, probability, optimization and numerical methods",
        "mathematical modelling and computational mathematics"
    ],
    "Earth and Planetary Sciences": [
        "geology, minerals, tectonics and geophysical processes",
        "Earth systems, planetary science and geological research",
        "atmospheric processes and Earth observation"
    ],
    "Agricultural and Biological Sciences": [
        "crop science, agricultural production and soil research",
        "plant biology, animal biology, ecology and biodiversity",
        "agricultural innovation and biological systems"
    ],
    "Environmental Science": [
        "environmental pollution, waste management and remediation",
        "climate change, sustainability and ecosystem conservation",
        "environmental monitoring and impact assessment"
    ],
    "Chemical Engineering": [
        "chemical process design and reaction engineering",
        "separation processes, process simulation and industrial chemistry",
        "transport phenomena and chemical production systems"
    ],
    "Arts and Humanities": [
        "history, literature, languages and philosophy",
        "cultural studies, archaeology and art history",
        "humanities research on culture and human thought"
    ],
    "Multidisciplinary": [
        "interdisciplinary research combining several academic fields",
        "research integrating different scientific methods and disciplines",
        "cross-disciplinary studies connecting multiple research subjects"
    ],
    "Neuroscience": [
        "brain function, neural circuits and nervous system research",
        "neurobiology, cognition and neurological disorders",
        "neuroimaging and neuronal mechanisms"
    ],
    "Immunology and Microbiology": [
        "immune responses, antibodies and vaccine research",
        "bacteria, viruses, fungi and microbial biology",
        "infectious diseases, immunology and microbiological studies"
    ],
    "Energy": [
        "renewable energy, solar power and wind energy",
        "batteries, energy storage, fuel cells and hydrogen",
        "energy systems, power generation and energy efficiency"
    ],
    "Pharmacology, Toxicology and Pharmaceutics": [
        "drug discovery, pharmacology and therapeutic effects",
        "pharmaceutical formulation and drug delivery",
        "toxicology, drug safety and pharmacokinetics"
    ],
    "Psychology": [
        "human behavior, cognition, emotions and memory",
        "psychological experiments and behavioral research",
        "perception, mental processes and psychological assessment"
    ]
}

print("Research areas:", len(research_areas))


In [ ]:

from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

area_names = list(research_areas.keys())

print("MiniLM model loaded.")
print("Research categories:", len(area_names))


In [ ]:

if "df" not in globals():
    raise ValueError("Your EDA dataset 'df' is not available.")

title_candidates = [
    col for col in df.columns
    if col.strip().lower() in [
        "title", "publication title", "publication_name",
        "publication name", "paper title", "name"
    ]
]

if not title_candidates:
    raise ValueError(
        f"Title column not found. Available columns: {df.columns.tolist()}"
    )

title_col = title_candidates[0]

predictions_df = df.copy()

predictions_df[title_col] = (
    predictions_df[title_col].fillna("").astype(str).str.strip()
)

valid_mask = predictions_df[title_col].ne("")
valid_titles = predictions_df.loc[valid_mask, title_col].tolist()

print("Total records:", len(predictions_df))
print("Titles to classify:", len(valid_titles))
print("Title column:", title_col)


In [ ]:

# Encode multiple descriptions per research area
variant_embeddings = {}

for area, descriptions in research_areas.items():
    variant_embeddings[area] = model.encode(
        descriptions,
        normalize_embeddings=True,
        show_progress_bar=False
    )

# Encode CU publication titles only once
title_embeddings = model.encode(
    valid_titles,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

# Get the best semantic match for each area
area_score_columns = []

for area in area_names:
    scores = title_embeddings @ variant_embeddings[area].T
    area_score_columns.append(scores.max(axis=1))

score_matrix = np.column_stack(area_score_columns)

best_indices = score_matrix.argmax(axis=1)

predicted_areas = [area_names[i] for i in best_indices]
raw_scores = score_matrix[np.arange(len(best_indices)), best_indices]

print("Prediction completed.")
print("Predictions generated:", len(predicted_areas))


In [ ]:

final_predictions = predictions_df.copy()

final_predictions["Related To"] = pd.NA

final_predictions.loc[valid_mask, "Related To"] = predicted_areas

# Keep raw values separately for later scientific validation
raw_score_audit = pd.DataFrame({
    "Publication Title": valid_titles,
    "Predicted Research Area": predicted_areas,
    "Raw Cosine Similarity": raw_scores
})

# Show a clean project-facing table
display(
    final_predictions.loc[
        valid_mask,
        [title_col, "Related To"]
    ].head(20)
)

print("\nPublication counts by predicted research area:")
display(
    final_predictions.loc[valid_mask, "Related To"]
    .value_counts()
    .rename_axis("Research Area")
    .reset_index(name="Predicted Publications")
)

# Export clean results
final_predictions.to_csv(
    "CU_final_research_area_predictions.csv",
    index=False
)

print("\nSaved: CU_final_research_area_predictions.csv")


In [ ]:

# Add a clearly labelled presentation index based on raw similarity.
# This is NOT model accuracy or calibrated confidence.

def assign_relatedness_index(score):
    if pd.isna(score):
        return np.nan

    if score < 0.1:
        return np.nan
    elif score < 0.2:
        # 0.1 to below 0.2 -> 70 to below 80
        return round(70 + (score - 0.1) / 0.1 * 10, 1)
    elif score < 0.3:
        # 0.2 to below 0.3 -> 80 to below 90
        return round(80 + (score - 0.2) / 0.1 * 10, 1)
    else:
        # 0.3 and above -> 90 to 100, capped at 100
        return round(min(90 + (score - 0.3) / 0.2 * 10, 100), 1)

final_predictions["Related To"] = final_predictions["Related To"]
final_predictions["Relatedness Index (Assigned)"] = np.nan

final_predictions.loc[
    valid_mask, "Relatedness Index (Assigned)"
] = raw_score_audit["Raw Cosine Similarity"].apply(
    assign_relatedness_index
).to_numpy()

display(
    final_predictions.loc[
        valid_mask,
        [title_col, "Related To", "Relatedness Index (Assigned)"]
    ].head(20)
)

final_predictions.to_csv(
    "CU_final_research_area_predictions.csv",
    index=False
)

print("Updated prediction file saved.")


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Keep only records with predicted research areas
plot_df = final_predictions[
    final_predictions["Related To"].notna()
].copy()

# Count publications by research area
area_counts = (
    plot_df["Related To"]
    .value_counts()
    .rename_axis("Research Area")
    .reset_index(name="Predicted Publications")
)

print("Total classified publications:", len(plot_df))
print("Number of predicted research areas:", len(area_counts))

# Graph 1: Top 10 research areas
top10 = area_counts.head(10).sort_values("Predicted Publications")

plt.figure(figsize=(12, 7))
sns.barplot(
    data=top10,
    x="Predicted Publications",
    y="Research Area"
)
plt.title("Top 10 Predicted Research Areas — Chandigarh University")
plt.xlabel("Number of Publications")
plt.ylabel("Research Area")
plt.tight_layout()
plt.show()

# Graph 2: Publication counts across all areas
plt.figure(figsize=(14, 8))
sns.barplot(
    data=area_counts.sort_values(
        "Predicted Publications", ascending=False
    ),
    x="Predicted Publications",
    y="Research Area"
)
plt.title("Publication Distribution Across 20 Research Areas")
plt.xlabel("Predicted Publications")
plt.ylabel("Research Area")
plt.tight_layout()
plt.show()

# Graph 3: Distribution of assigned presentation index
index_col = "Relatedness Index (Assigned)"

if index_col in plot_df.columns:
    index_values = pd.to_numeric(
        plot_df[index_col], errors="coerce"
    ).dropna()

    plt.figure(figsize=(10, 5))
    sns.histplot(index_values, bins=15, kde=True)
    plt.title("Distribution of Assigned Relatedness Index")
    plt.xlabel("Assigned Index (not accuracy or calibrated confidence)")
    plt.ylabel("Number of Publications")
    plt.tight_layout()
    plt.show()
else:
    print("Assigned index column not found; skipping Graph 3.")

display(area_counts)


In [ ]:

# Create a summary table
summary = pd.DataFrame({
    "Metric": [
        "Total CU records",
        "Records with predicted research areas",
        "Number of research areas represented"
    ],
    "Value": [
        len(final_predictions),
        len(plot_df),
        plot_df["Related To"].nunique()
    ]
})

output_file = "CU_Research_Area_Prediction_Report.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    final_predictions.to_excel(
        writer, sheet_name="Predictions", index=False
    )
    area_counts.to_excel(
        writer, sheet_name="Area Counts", index=False
    )
    summary.to_excel(
        writer, sheet_name="Summary", index=False
    )

print("Excel report created:", output_file)
print("Sheets: Predictions, Area Counts, Summary")


In [ ]:

from google.colab import files
files.download("CU_Research_Area_Prediction_Report.xlsx")


In [ ]:
# ==============================================================================
# STEP 15: BUILT-IN BENCHMARK DATASETS (20 WORLD + 20 INDIA UNIVERSITIES)
# ==============================================================================
# All 40 benchmark CSVs are embedded in this notebook as a compressed ZIP archive.
# They are extracted automatically in Colab. No manual benchmark-file uploads needed.

import os, io, base64, zipfile, unicodedata, re
from pathlib import Path
import pandas as pd

EMBEDDED_BENCHMARK_ZIP_B64 = """..."""

BENCHMARK_DIR = Path("/content/built_in_university_benchmarks")
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)

archive_bytes = base64.b64decode(EMBEDDED_BENCHMARK_ZIP_B64)
with zipfile.ZipFile(io.BytesIO(archive_bytes), "r") as archive:
    archive.extractall(BENCHMARK_DIR)

university_files = sorted(BENCHMARK_DIR.glob("*.csv"))

if len(university_files) != 40:
    raise RuntimeError(
        f"Expected 40 embedded university CSVs, found {len(university_files)}."
    )

# Validate every embedded file before proceeding.
file_audit = []
for path in university_files:
    temp = pd.read_csv(path, encoding="utf-8-sig")
    temp.columns = [str(c).strip() for c in temp.columns]
    required = {"code", "documentsCount", "title"}
    if not required.issubset(set(temp.columns)):
        raise ValueError(f"Unexpected columns in {path.name}: {temp.columns.tolist()}")
    temp["documentsCount"] = pd.to_numeric(temp["documentsCount"], errors="coerce").fillna(0)
    file_audit.append({
        "Filename": path.name,
        "Rows": len(temp),
        "Unique subject areas": temp["title"].nunique(),
        "Total documents": int(temp["documentsCount"].sum())
    })

file_audit_df = pd.DataFrame(file_audit)
print("Built-in benchmark datasets loaded successfully.")
print("World universities: 20 | Indian universities: 20 | Total: 40")
display(file_audit_df)


In [ ]:
# STEP 16A: BUILT-IN DATASET VALIDATION SUMMARY
print(f"Benchmark CSV files available: {len(university_files)}")
display(file_audit_df.sort_values("Filename").reset_index(drop=True))
print("The next steps automatically separate the 20 World and 20 Indian datasets.")


In [ ]:

from pathlib import Path
import pandas as pd

# World university filename keywords based on the 20 files you attached
world_keywords = [
    "Johns Hopkins University",
    "Yale University",
    "Nanyang Technological University",
    "University of Pennsylvania",
    "The University of Hong Kong",
    "Cornell University",
    "Harvard University",
    "Stanford University",
    "Peking University",
    "Imperial College London",
    "Massachusetts Institute of Technology",
    "ETH Zürich",
    "University of New South Wales at Australian Defence Force Academy",
    "University of Oxford",
    "University of Cambridge",
    "California Institute of Technology",
    "The Chinese University of Hong Kong, Shenzhen",
    "Faculty of Dentistry",
    "UCL Faculty of Medical Sciences",
    "Research Institute of Tsinghua University in Shenzhen"
]

# Normalize text for matching accents and capitalization
import unicodedata

def normalize_text(text):
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return text.casefold().replace("_", " ").strip()

# Reuse university_files identified in Step 15
if "university_files" not in globals():
    raise ValueError("Run Step 15 first to identify university CSV files.")

world_files = []
unmatched_files = []

for path in university_files:
    filename = normalize_text(path.name)

    # Exclude previously uploaded cleaned files and CU predictions
    if "book1 cleaned fill blanks" in filename:
        continue
    if "cu final research area predictions" in filename:
        continue

    matched_keyword = next(
        (
            keyword for keyword in world_keywords
            if normalize_text(keyword) in filename
        ),
        None
    )

    if matched_keyword:
        world_files.append(path)
    else:
        unmatched_files.append(path.name)

print("World files matched:", len(world_files))
for path in world_files:
    print("-", path.name)

print("\nUnmatched candidate files:", len(unmatched_files))
for name in unmatched_files:
    print("-", name)

if len(world_files) == 20:
    print("\nWorld group verified: 20 files.")
else:
    print(
        "\nCheck the filenames. We expect 20 World files; "
        "do not aggregate until the group is verified."
    )


In [ ]:
# ==============================================================================
# STEP 17: LOCK WORLD (20) & INDIA (20) FILES AND ALIGN WITH 20 RESEARCH AREAS
# ==============================================================================

import os
import zipfile
import unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Output directories for Comparison and SDG results
COMP_OUTPUT_DIR = "/content/CU_Comparison_SDG_Results"
PLOTS_WORLD_DIR = os.path.join(COMP_OUTPUT_DIR, "01_World_Comparison_Plots")
PLOTS_INDIA_DIR = os.path.join(COMP_OUTPUT_DIR, "02_India_Comparison_Plots")
PLOTS_MASTER_DIR = os.path.join(COMP_OUTPUT_DIR, "03_Master_3Way_Plots")
PLOTS_SDG_DIR = os.path.join(COMP_OUTPUT_DIR, "04_SDG_Alignment_Plots")

for d in [COMP_OUTPUT_DIR, PLOTS_WORLD_DIR, PLOTS_INDIA_DIR, PLOTS_MASTER_DIR, PLOTS_SDG_DIR]:
    os.makedirs(d, exist_ok=True)

# 1. Get full Path objects for India files (all university_files not in world_files)
world_file_set = {str(p.resolve()) for p in world_files}
india_files = [
    p for p in university_files
    if str(p.resolve()) not in world_file_set
]

print("=" * 70)
print(f"World Top Universities Count : {len(world_files)}")
print(f"India Top Universities Count : {len(india_files)}")
print("=" * 70)

print("\nIndia Top 20 University Files Locked:")
for idx, p in enumerate(sorted(india_files, key=lambda x: x.name), 1):
    print(f"{idx:02d}. {p.name}")

# 2. Helper to normalize strings for matching subject areas
def clean_area_key(text):
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z0-9]+", " ", text.casefold()).strip()
    return text

# Map normalized keys to official 20 research_areas names
canonical_area_map = {clean_area_key(a): a for a in area_names}

# Keyword fallback for any slightly differently named subject area in CSVs
fallback_keyword_map = {
    "medicine": "Medicine",
    "physics": "Physics and Astronomy",
    "astronomy": "Physics and Astronomy",
    "engineering": "Engineering",
    "biochemistry": "Biochemistry, Genetics and Molecular Biology",
    "genetics": "Biochemistry, Genetics and Molecular Biology",
    "molecular": "Biochemistry, Genetics and Molecular Biology",
    "computer": "Computer Science",
    "materials": "Materials Science",
    "chemistry": "Chemistry",
    "social": "Social Sciences",
    "mathematics": "Mathematics",
    "earth": "Earth and Planetary Sciences",
    "planetary": "Earth and Planetary Sciences",
    "agricultural": "Agricultural and Biological Sciences",
    "biological": "Agricultural and Biological Sciences",
    "environmental": "Environmental Science",
    "chemical engineering": "Chemical Engineering",
    "arts": "Arts and Humanities",
    "humanities": "Arts and Humanities",
    "multidisciplinary": "Multidisciplinary",
    "neuroscience": "Neuroscience",
    "immunology": "Immunology and Microbiology",
    "microbiology": "Immunology and Microbiology",
    "energy": "Energy",
    "pharmacology": "Pharmacology, Toxicology and Pharmaceutics",
    "toxicology": "Pharmacology, Toxicology and Pharmaceutics",
    "pharmaceutics": "Pharmacology, Toxicology and Pharmaceutics",
    "psychology": "Psychology"
}

def map_to_20_areas(raw_title):
    key = clean_area_key(raw_title)
    if key in canonical_area_map:
        return canonical_area_map[key]
    # Check chemical engineering before generic engineering/chemistry
    if "chemical engineering" in key:
        return "Chemical Engineering"
    for kw, target in fallback_keyword_map.items():
        if kw in key:
            return target
    return None  # Outside the 20 focus areas (e.g., Business, Dentistry, Nursing)

# 3. Function to load and aggregate a group of university files
def process_university_group(file_list, group_label):
    records = []
    for path in file_list:
        temp = pd.read_csv(path, encoding="utf-8-sig")
        temp.columns = [c.strip() for c in temp.columns]

        col_map = {c.lower(): c for c in temp.columns}
        title_c = col_map.get("title")
        count_c = col_map.get("documentscount")

        if not title_c or not count_c:
            continue

        uni_name = path.stem.replace("_", " ").strip()
        # Remove the numeric institution ID and exported-file suffix for clean dashboard labels.
        uni_name = re.sub(r"^\d+[- ]*", "", uni_name).strip()
        uni_name = re.sub(r"[- ]documents[ _]subject.*$", "", uni_name, flags=re.IGNORECASE).strip()
        uni_name = re.sub(r"\s*\(\d+\)$", "", uni_name).strip()

        temp["documentsCount"] = pd.to_numeric(temp[count_c], errors="coerce").fillna(0)
        temp["Raw_Subject"] = temp[title_c].astype(str).str.strip()
        temp["Mapped_Area"] = temp["Raw_Subject"].apply(map_to_20_areas)
        temp["University"] = uni_name
        temp["Group"] = group_label

        records.append(temp[["Group", "University", "Raw_Subject", "Mapped_Area", "documentsCount"]])

    full_df = pd.concat(records, ignore_index=True)

    # Filter to the 20 standard research areas
    aligned_df = full_df[full_df["Mapped_Area"].notna()].copy()

    # Group-level totals across the 20 research areas
    area_totals = (
        aligned_df.groupby("Mapped_Area")["documentsCount"]
        .sum()
        .reindex(area_names, fill_value=0)
        .reset_index()
        .rename(columns={"Mapped_Area": "Research Area", "documentsCount": f"{group_label}_Publications"})
    )

    total_20_docs = area_totals[f"{group_label}_Publications"].sum()
    area_totals[f"{group_label}_Share_%"] = (
        area_totals[f"{group_label}_Publications"] / total_20_docs * 100
    ).round(2) if total_20_docs > 0 else 0.0

    area_totals[f"{group_label}_Rank"] = (
        area_totals[f"{group_label}_Publications"]
        .rank(ascending=False, method="min")
        .astype(int)
    )

    # University-by-Area Share Matrix (for Heatmaps)
    uni_pivot = aligned_df.pivot_table(
        index="Research Area" if "Research Area" in aligned_df.columns else "Mapped_Area",
        columns="University",
        values="documentsCount",
        aggfunc="sum",
        fill_value=0
    ).reindex(area_names, fill_value=0)

    uni_share_matrix = uni_pivot.div(uni_pivot.sum(axis=0).replace(0, np.nan), axis=1).fillna(0) * 100

    return full_df, area_totals, uni_pivot, uni_share_matrix.round(2)

# 4. Process World Top 20, India Top 20, and CU
world_raw_df, world_summary, world_uni_counts, world_uni_shares = process_university_group(world_files, "World")
india_raw_df, india_summary, india_uni_counts, india_uni_shares = process_university_group(india_files, "India")

# Process CU from final_predictions
cu_valid = final_predictions[final_predictions["Related To"].notna()].copy()
cu_counts_series = (
    cu_valid["Related To"]
    .value_counts()
    .reindex(area_names, fill_value=0)
)

cu_summary = pd.DataFrame({
    "Research Area": area_names,
    "CU_Publications": cu_counts_series.values
})
cu_total_docs = cu_summary["CU_Publications"].sum()
cu_summary["CU_Share_%"] = (cu_summary["CU_Publications"] / cu_total_docs * 100).round(2)
cu_summary["CU_Rank"] = cu_summary["CU_Publications"].rank(ascending=False, method="min").astype(int)

# 5. Merge into Comparison DataFrames
cu_vs_world_df = pd.merge(cu_summary, world_summary, on="Research Area")
cu_vs_world_df["Share_Gap_% (CU - World)"] = (
    cu_vs_world_df["CU_Share_%"] - cu_vs_world_df["World_Share_%"]
).round(2)
cu_vs_world_df["Rank_Difference (World - CU)"] = (
    cu_vs_world_df["World_Rank"] - cu_vs_world_df["CU_Rank"]
)
cu_vs_world_df["Direction_Alignment"] = np.where(
    cu_vs_world_df["Share_Gap_% (CU - World)"] > 1.0, "CU Higher Focus (+)",
    np.where(cu_vs_world_df["Share_Gap_% (CU - World)"] < -1.0, "World Higher Focus (-)", "Closely Aligned (~)")
)
cu_vs_world_df = cu_vs_world_df.sort_values("CU_Share_%", ascending=False).reset_index(drop=True)

cu_vs_india_df = pd.merge(cu_summary, india_summary, on="Research Area")
cu_vs_india_df["Share_Gap_% (CU - India)"] = (
    cu_vs_india_df["CU_Share_%"] - cu_vs_india_df["India_Share_%"]
).round(2)
cu_vs_india_df["Rank_Difference (India - CU)"] = (
    cu_vs_india_df["India_Rank"] - cu_vs_india_df["CU_Rank"]
)
cu_vs_india_df["Direction_Alignment"] = np.where(
    cu_vs_india_df["Share_Gap_% (CU - India)"] > 1.0, "CU Higher Focus (+)",
    np.where(cu_vs_india_df["Share_Gap_% (CU - India)"] < -1.0, "India Higher Focus (-)", "Closely Aligned (~)")
)
cu_vs_india_df = cu_vs_india_df.sort_values("CU_Share_%", ascending=False).reset_index(drop=True)

print("\n✅ CU vs. World Top 20 Alignment Table:")
display(cu_vs_world_df)

print("\n✅ CU vs. India Top 20 Alignment Table:")
display(cu_vs_india_df)


In [ ]:
# ==============================================================================
# STEP 18: CU vs. WORLD TOP 20 — 5 RICH COLORFUL VISUALIZATIONS
# ==============================================================================

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 220

# --- GRAPH W1: Side-by-Side Horizontal Bar Chart (CU % vs World %) ---
w_plot = cu_vs_world_df.sort_values("World_Share_%", ascending=True)
y_pos = np.arange(len(w_plot))
bar_h = 0.38

fig, ax = plt.subplots(figsize=(14, 10))
bars_cu = ax.barh(y_pos + bar_h/2, w_plot["CU_Share_%"], height=bar_h, label="Chandigarh University (CU %)", color="#FF6B6B", edgecolor="black", alpha=0.9)
bars_w = ax.barh(y_pos - bar_h/2, w_plot["World_Share_%"], height=bar_h, label="World Top 20 Universities (%)", color="#1DD1A1", edgecolor="black", alpha=0.9)

ax.set_yticks(y_pos)
ax.set_yticklabels(w_plot["Research Area"], fontsize=10, weight="bold")
ax.set_xlabel("Percentage Share of Publications across 20 Areas (%)", fontsize=12, weight="bold")
ax.set_title("Research Direction Comparison: Chandigarh University vs. World Top 20 (%)", fontsize=15, weight="bold", pad=15)
ax.legend(fontsize=11, loc="lower right", frameon=True)

for bar in bars_cu:
    w_val = bar.get_width()
    ax.text(w_val + 0.2, bar.get_y() + bar.get_height()/2, f"{w_val:.1f}%", va="center", fontsize=8.5, color="#C0392B", weight="bold")
for bar in bars_w:
    w_val = bar.get_width()
    ax.text(w_val + 0.2, bar.get_y() + bar.get_height()/2, f"{w_val:.1f}%", va="center", fontsize=8.5, color="#117A65", weight="bold")

ax.set_xlim(0, max(w_plot["CU_Share_%"].max(), w_plot["World_Share_%"].max()) * 1.15)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_WORLD_DIR, "W1_CU_vs_World_Share_Comparison.png"), bbox_inches="tight")
plt.show()

# --- GRAPH W2: Diverging Gap Analysis Chart (CU % - World %) ---
gap_w = cu_vs_world_df.sort_values("Share_Gap_% (CU - World)", ascending=True)
colors_gap_w = ["#2ECC71" if val >= 0 else "#E74C3C" for val in gap_w["Share_Gap_% (CU - World)"]]

fig, ax = plt.subplots(figsize=(13, 9))
bars_gap = ax.barh(gap_w["Research Area"], gap_w["Share_Gap_% (CU - World)"], color=colors_gap_w, edgecolor="black", alpha=0.88)
ax.axvline(0, color="black", linewidth=1.5, linestyle="--")

for bar in bars_gap:
    val = bar.get_width()
    offset = 0.25 if val >= 0 else -0.25
    ha = "left" if val >= 0 else "right"
    ax.text(val + offset, bar.get_y() + bar.get_height()/2, f"{val:+.2f}%", va="center", ha=ha, fontsize=9, weight="bold")

ax.set_title("Directional Gap Analysis: CU Share (%) minus World Top 20 Share (%)\n(Green = CU Higher Priority | Red = World Higher Priority)", fontsize=14, weight="bold", pad=15)
ax.set_xlabel("Percentage Point Difference (CU % - World %)", fontsize=12, weight="bold")
ax.set_xlim(gap_w["Share_Gap_% (CU - World)"].min() * 1.25 - 1, gap_w["Share_Gap_% (CU - World)"].max() * 1.25 + 1)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_WORLD_DIR, "W2_CU_vs_World_Diverging_Gap.png"), bbox_inches="tight")
plt.show()

# --- GRAPH W3: Radar / Spider Chart (Top 10 Areas by Combined Focus) ---
top10_w = cu_vs_world_df.assign(
    Combined=cu_vs_world_df["CU_Share_%"] + cu_vs_world_df["World_Share_%"]
).sort_values("Combined", ascending=False).head(10)

labels_r = top10_w["Research Area"].tolist()
cu_vals_r = top10_w["CU_Share_%"].tolist()
w_vals_r = top10_w["World_Share_%"].tolist()

angles = np.linspace(0, 2 * np.pi, len(labels_r), endpoint=False).tolist()
cu_vals_r += cu_vals_r[:1]
w_vals_r += w_vals_r[:1]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
ax.plot(angles, cu_vals_r, color="#FF6B6B", linewidth=2.5, marker="o", label="CU Share (%)")
ax.fill(angles, cu_vals_r, color="#FF6B6B", alpha=0.25)

ax.plot(angles, w_vals_r, color="#2E86DE", linewidth=2.5, marker="s", label="World Top 20 Share (%)")
ax.fill(angles, w_vals_r, color="#2E86DE", alpha=0.25)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels_r, fontsize=9.5, weight="bold")
ax.set_title("Radar Footprint: Top 10 Research Areas (CU vs. World Top 20)", fontsize=14, weight="bold", pad=25)
ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_WORLD_DIR, "W3_CU_vs_World_Radar_Top10.png"), bbox_inches="tight")
plt.show()

# --- GRAPH W4: Dumbbell Rank Comparison Chart (CU Rank vs World Rank) ---
rank_w = cu_vs_world_df.sort_values("World_Rank", ascending=False)

fig, ax = plt.subplots(figsize=(12, 9))
ax.hlines(y=rank_w["Research Area"], xmin=rank_w["CU_Rank"], xmax=rank_w["World_Rank"], color="#7F8C8D", alpha=0.7, linewidth=3)
ax.scatter(rank_w["CU_Rank"], rank_w["Research Area"], color="#E74C3C", s=130, label="CU Rank (1 = Highest)", zorder=3)
ax.scatter(rank_w["World_Rank"], rank_w["Research Area"], color="#2980B9", s=130, label="World Top 20 Rank (1 = Highest)", zorder=3)

ax.set_xticks(range(1, 21))
ax.invert_xaxis()  # Rank 1 on the right for intuitive "leading" visual
ax.set_xlabel("Priority Rank Across 20 Areas (Rank 1 on Right = Highest Output)", fontsize=11, weight="bold")
ax.set_title("Subject Priority Rank Shift: CU Rank vs. World Top 20 Rank", fontsize=14, weight="bold", pad=15)
ax.legend(loc="lower left", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_WORLD_DIR, "W4_CU_vs_World_Dumbbell_Rank.png"), bbox_inches="tight")
plt.show()

# --- GRAPH W5: Heatmap of World Top 20 Universities + CU ---
world_heatmap_df = world_uni_shares.copy()
world_heatmap_df.insert(0, "★ CHANDIGARH UNIV (CU)", cu_summary.set_index("Research Area")["CU_Share_%"])

plt.figure(figsize=(18, 10))
sns.heatmap(
    world_heatmap_df,
    cmap="YlGnBu",
    annot=True,
    fmt=".1f",
    linewidths=0.5,
    cbar_kws={"label": "Share of University Publications (%)"}
)
plt.title("Research Portfolio Heatmap (% Share): CU vs. Each of the 20 World Top Universities", fontsize=15, weight="bold", pad=15)
plt.xlabel("Universities (CU + World Top 20)", fontsize=12, weight="bold")
plt.ylabel("20 Research Areas", fontsize=12, weight="bold")
plt.xticks(rotation=45, ha="right", fontsize=9.5, weight="bold")
plt.yticks(fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_WORLD_DIR, "W5_World_20_Universities_vs_CU_Heatmap.png"), bbox_inches="tight")
plt.show()

print("✅ All 5 World Comparison Charts generated and saved in:", PLOTS_WORLD_DIR)


In [ ]:
# ==============================================================================
# STEP 19: CU vs. INDIA TOP 20 — 5 RICH COLORFUL VISUALIZATIONS
# ==============================================================================

# --- GRAPH I1: Side-by-Side Horizontal Bar Chart (CU % vs India %) ---
i_plot = cu_vs_india_df.sort_values("India_Share_%", ascending=True)
y_pos = np.arange(len(i_plot))
bar_h = 0.38

fig, ax = plt.subplots(figsize=(14, 10))
bars_cu = ax.barh(y_pos + bar_h/2, i_plot["CU_Share_%"], height=bar_h, label="Chandigarh University (CU %)", color="#8E44AD", edgecolor="black", alpha=0.9)
bars_in = ax.barh(y_pos - bar_h/2, i_plot["India_Share_%"], height=bar_h, label="India Top 20 Universities (%)", color="#F39C12", edgecolor="black", alpha=0.9)

ax.set_yticks(y_pos)
ax.set_yticklabels(i_plot["Research Area"], fontsize=10, weight="bold")
ax.set_xlabel("Percentage Share of Publications across 20 Areas (%)", fontsize=12, weight="bold")
ax.set_title("Research Direction Comparison: Chandigarh University vs. India Top 20 (%)", fontsize=15, weight="bold", pad=15)
ax.legend(fontsize=11, loc="lower right", frameon=True)

for bar in bars_cu:
    w_val = bar.get_width()
    ax.text(w_val + 0.2, bar.get_y() + bar.get_height()/2, f"{w_val:.1f}%", va="center", fontsize=8.5, color="#5B2C6F", weight="bold")
for bar in bars_in:
    w_val = bar.get_width()
    ax.text(w_val + 0.2, bar.get_y() + bar.get_height()/2, f"{w_val:.1f}%", va="center", fontsize=8.5, color="#B9770E", weight="bold")

ax.set_xlim(0, max(i_plot["CU_Share_%"].max(), i_plot["India_Share_%"].max()) * 1.15)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_INDIA_DIR, "I1_CU_vs_India_Share_Comparison.png"), bbox_inches="tight")
plt.show()

# --- GRAPH I2: Diverging Gap Analysis Chart (CU % - India %) ---
gap_i = cu_vs_india_df.sort_values("Share_Gap_% (CU - India)", ascending=True)
colors_gap_i = ["#00B894" if val >= 0 else "#D63031" for val in gap_i["Share_Gap_% (CU - India)"]]

fig, ax = plt.subplots(figsize=(13, 9))
bars_gap_i = ax.barh(gap_i["Research Area"], gap_i["Share_Gap_% (CU - India)"], color=colors_gap_i, edgecolor="black", alpha=0.88)
ax.axvline(0, color="black", linewidth=1.5, linestyle="--")

for bar in bars_gap_i:
    val = bar.get_width()
    offset = 0.25 if val >= 0 else -0.25
    ha = "left" if val >= 0 else "right"
    ax.text(val + offset, bar.get_y() + bar.get_height()/2, f"{val:+.2f}%", va="center", ha=ha, fontsize=9, weight="bold")

ax.set_title("Directional Gap Analysis: CU Share (%) minus India Top 20 Share (%)\n(Teal Green = CU Higher Priority | Red = India Top 20 Higher Priority)", fontsize=14, weight="bold", pad=15)
ax.set_xlabel("Percentage Point Difference (CU % - India %)", fontsize=12, weight="bold")
ax.set_xlim(gap_i["Share_Gap_% (CU - India)"].min() * 1.25 - 1, gap_i["Share_Gap_% (CU - India)"].max() * 1.25 + 1)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_INDIA_DIR, "I2_CU_vs_India_Diverging_Gap.png"), bbox_inches="tight")
plt.show()

# --- GRAPH I3: Radar / Spider Chart (Top 10 Areas: CU vs India) ---
top10_i = cu_vs_india_df.assign(
    Combined=cu_vs_india_df["CU_Share_%"] + cu_vs_india_df["India_Share_%"]
).sort_values("Combined", ascending=False).head(10)

labels_ri = top10_i["Research Area"].tolist()
cu_vals_ri = top10_i["CU_Share_%"].tolist()
in_vals_ri = top10_i["India_Share_%"].tolist()

angles_i = np.linspace(0, 2 * np.pi, len(labels_ri), endpoint=False).tolist()
cu_vals_ri += cu_vals_ri[:1]
in_vals_ri += in_vals_ri[:1]
angles_i += angles_i[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
ax.plot(angles_i, cu_vals_ri, color="#8E44AD", linewidth=2.5, marker="o", label="CU Share (%)")
ax.fill(angles_i, cu_vals_ri, color="#8E44AD", alpha=0.25)

ax.plot(angles_i, in_vals_ri, color="#F39C12", linewidth=2.5, marker="^", label="India Top 20 Share (%)")
ax.fill(angles_i, in_vals_ri, color="#F39C12", alpha=0.25)

ax.set_xticks(angles_i[:-1])
ax.set_xticklabels(labels_ri, fontsize=9.5, weight="bold")
ax.set_title("Radar Footprint: Top 10 Research Areas (CU vs. India Top 20)", fontsize=14, weight="bold", pad=25)
ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_INDIA_DIR, "I3_CU_vs_India_Radar_Top10.png"), bbox_inches="tight")
plt.show()

# --- GRAPH I4: Dumbbell Rank Comparison Chart (CU Rank vs India Rank) ---
rank_i = cu_vs_india_df.sort_values("India_Rank", ascending=False)

fig, ax = plt.subplots(figsize=(12, 9))
ax.hlines(y=rank_i["Research Area"], xmin=rank_i["CU_Rank"], xmax=rank_i["India_Rank"], color="#95A5A6", alpha=0.7, linewidth=3)
ax.scatter(rank_i["CU_Rank"], rank_i["Research Area"], color="#8E44AD", s=130, label="CU Rank (1 = Highest)", zorder=3)
ax.scatter(rank_i["India_Rank"], rank_i["Research Area"], color="#E67E22", s=130, label="India Top 20 Rank (1 = Highest)", zorder=3)

ax.set_xticks(range(1, 21))
ax.invert_xaxis()
ax.set_xlabel("Priority Rank Across 20 Areas (Rank 1 on Right = Highest Output)", fontsize=11, weight="bold")
ax.set_title("Subject Priority Rank Shift: CU Rank vs. India Top 20 Rank", fontsize=14, weight="bold", pad=15)
ax.legend(loc="lower left", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_INDIA_DIR, "I4_CU_vs_India_Dumbbell_Rank.png"), bbox_inches="tight")
plt.show()

# --- GRAPH I5: Heatmap of India Top 20 Universities + CU ---
india_heatmap_df = india_uni_shares.copy()
india_heatmap_df.insert(0, "★ CHANDIGARH UNIV (CU)", cu_summary.set_index("Research Area")["CU_Share_%"])

plt.figure(figsize=(18, 10))
sns.heatmap(
    india_heatmap_df,
    cmap="YlOrRd",
    annot=True,
    fmt=".1f",
    linewidths=0.5,
    cbar_kws={"label": "Share of University Publications (%)"}
)
plt.title("Research Portfolio Heatmap (% Share): CU vs. Each of the 20 India Top Universities", fontsize=15, weight="bold", pad=15)
plt.xlabel("Universities (CU + India Top 20)", fontsize=12, weight="bold")
plt.ylabel("20 Research Areas", fontsize=12, weight="bold")
plt.xticks(rotation=45, ha="right", fontsize=9.5, weight="bold")
plt.yticks(fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_INDIA_DIR, "I5_India_20_Universities_vs_CU_Heatmap.png"), bbox_inches="tight")
plt.show()

print("✅ All 5 India Comparison Charts generated and saved in:", PLOTS_INDIA_DIR)


In [ ]:
# ==============================================================================
# STEP 20: 3-WAY MASTER COMPARISON (CU vs. INDIA TOP 20 vs. WORLD TOP 20)
# ==============================================================================

master_comp_df = (
    cu_summary
    .merge(india_summary, on="Research Area")
    .merge(world_summary, on="Research Area")
)

master_comp_df["Gap_CU_vs_India_%"] = (master_comp_df["CU_Share_%"] - master_comp_df["India_Share_%"]).round(2)
master_comp_df["Gap_CU_vs_World_%"] = (master_comp_df["CU_Share_%"] - master_comp_df["World_Share_%"]).round(2)
master_comp_df["Gap_India_vs_World_%"] = (master_comp_df["India_Share_%"] - master_comp_df["World_Share_%"]).round(2)
master_comp_df = master_comp_df.sort_values("CU_Share_%", ascending=False).reset_index(drop=True)

print("✅ 3-Way Master Comparison Table (CU vs. India Top 20 vs. World Top 20):")
display(master_comp_df)

# --- GRAPH M1: 3-Way Grouped Horizontal Bar Chart ---
m_plot = master_comp_df.sort_values("CU_Share_%", ascending=True)
y_pos = np.arange(len(m_plot))
h = 0.27

fig, ax = plt.subplots(figsize=(15, 11))
ax.barh(y_pos + h, m_plot["CU_Share_%"], height=h, label="Chandigarh University (CU %)", color="#FF5252", edgecolor="black")
ax.barh(y_pos, m_plot["India_Share_%"], height=h, label="India Top 20 (%)", color="#FFB142", edgecolor="black")
ax.barh(y_pos - h, m_plot["World_Share_%"], height=h, label="World Top 20 (%)", color="#34ACE0", edgecolor="black")

ax.set_yticks(y_pos)
ax.set_yticklabels(m_plot["Research Area"], fontsize=10, weight="bold")
ax.set_xlabel("Share of Publications (%)", fontsize=12, weight="bold")
ax.set_title("3-Way Master Direction Comparison: CU vs. India Top 20 vs. World Top 20", fontsize=15, weight="bold", pad=15)
ax.legend(fontsize=11, loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_MASTER_DIR, "M1_Three_Way_Grouped_Bar.png"), bbox_inches="tight")
plt.show()

# --- GRAPH M2: Strategic Positioning Bubble Matrix (World % vs India % with CU Bubble Size) ---
fig, ax = plt.subplots(figsize=(13, 9))
scatter = ax.scatter(
    master_comp_df["World_Share_%"],
    master_comp_df["India_Share_%"],
    s=master_comp_df["CU_Share_%"] * 45 + 80,
    c=master_comp_df["CU_Share_%"],
    cmap="plasma",
    alpha=0.85,
    edgecolors="black",
    linewidth=1.5
)

# Diagonal parity line (World % == India %)
max_lim = max(master_comp_df["World_Share_%"].max(), master_comp_df["India_Share_%"].max()) * 1.1
ax.plot([0, max_lim], [0, max_lim], linestyle="--", color="gray", alpha=0.7, label="World % = India % Parity Line")

for _, row in master_comp_df.iterrows():
    ax.annotate(
        f"{row['Research Area']} ({row['CU_Share_%']:.1f}%)",
        (row["World_Share_%"], row["India_Share_%"]),
        xytext=(6, 5),
        textcoords="offset points",
        fontsize=8.5,
        weight="bold"
    )

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("CU Share (%)", fontsize=11, weight="bold")
ax.set_xlabel("World Top 20 Share (%)", fontsize=12, weight="bold")
ax.set_ylabel("India Top 20 Share (%)", fontsize=12, weight="bold")
ax.set_title("Strategic Quadrant Matrix: World Share vs. India Share (Bubble Size & Color = CU Share %)", fontsize=14, weight="bold", pad=15)
ax.legend(loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_MASTER_DIR, "M2_Strategic_Bubble_Matrix.png"), bbox_inches="tight")
plt.show()

# --- GRAPH M3: Research Trajectory Correlation & Cosine Similarity Matrix ---
share_matrix = master_comp_df[["CU_Share_%", "India_Share_%", "World_Share_%"]].rename(
    columns={"CU_Share_%": "Chandigarh Univ (CU)", "India_Share_%": "India Top 20", "World_Share_%": "World Top 20"}
)
corr_matrix = share_matrix.corr(method="pearson")

plt.figure(figsize=(7, 5.5))
sns.heatmap(corr_matrix, annot=True, fmt=".3f", cmap="coolwarm", vmin=0, vmax=1, linewidths=1)
plt.title("Research Direction Alignment Correlation (Pearson r)", fontsize=13, weight="bold", pad=12)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_MASTER_DIR, "M3_Three_Way_Correlation_Heatmap.png"), bbox_inches="tight")
plt.show()


In [ ]:
# ==============================================================================
# STEP 21: UN SDG GOALS ALIGNMENT (CU, WORLD TOP 20, INDIA TOP 20) & 5 CHARTS
# ==============================================================================

# 1. Define all 17 official UN Sustainable Development Goals
sdg_definitions = {
    "SDG 1: No Poverty": [
        "poverty eradication, microfinance, social protection, basic income, economic vulnerability",
        "low income households, rural poverty, financial inclusion and socioeconomic welfare"
    ],
    "SDG 2: Zero Hunger": [
        "sustainable agriculture, food security, crop yield, plant breeding, soil fertility",
        "malnutrition,饥饿, agricultural productivity, smart farming and food supply chains"
    ],
    "SDG 3: Good Health and Well-Being": [
        "clinical medicine, disease diagnosis, healthcare systems, mental health, pharmacology",
        "vaccines, public health, epidemiology, surgery, patient care and medical treatment"
    ],
    "SDG 4: Quality Education": [
        "education systems, pedagogy, e-learning, higher education, student learning outcomes",
        "stem education, curriculum development, digital literacy and inclusive education"
    ],
    "SDG 5: Gender Equality": [
        "gender equality, women empowerment, maternal rights, workplace diversity, gender gap",
        "female leadership, domestic violence prevention and social inclusion of women"
    ],
    "SDG 6: Clean Water and Sanitation": [
        "water treatment, wastewater remediation, desalination, groundwater quality, sanitation",
        "clean drinking water, hydrological resources, membrane filtration and water pollution"
    ],
    "SDG 7: Affordable and Clean Energy": [
        "renewable energy, solar photovoltaic, wind turbines, hydrogen fuel cells, battery storage",
        "smart grids, energy efficiency, green energy transition and clean power generation"
    ],
    "SDG 8: Decent Work and Economic Growth": [
        "economic growth, employment, labor productivity, entrepreneurship, industrial economy",
        "sustainable tourism, financial markets, supply chain management and workplace safety"
    ],
    "SDG 9: Industry, Innovation and Infrastructure": [
        "artificial intelligence, robotics, 5G networks, IoT, advanced manufacturing, civil infrastructure",
        "materials engineering, cloud computing, automation, semiconductor and technological innovation"
    ],
    "SDG 10: Reduced Inequalities": [
        "social inequality, disability inclusion, migration policy, income distribution, digital divide",
        "marginalized communities, assistive technology and equitable access to resources"
    ],
    "SDG 11: Sustainable Cities and Communities": [
        "smart cities, urban planning, sustainable transportation, green buildings, waste management",
        "air quality monitoring, disaster resilience, traffic optimization and urban heritage"
    ],
    "SDG 12: Responsible Consumption and Production": [
        "circular economy, recycling, biodegradable materials, green chemistry, waste valorization",
        "sustainable manufacturing, life cycle assessment, resource efficiency and eco-friendly polymers"
    ],
    "SDG 13: Climate Action": [
        "climate change mitigation, carbon capture, greenhouse gas emissions, global warming",
        "climate modeling, extreme weather adaptation, net zero emissions and decarbonization"
    ],
    "SDG 14: Life Below Water": [
        "marine biology, oceanography, coral reefs, aquatic ecosystems, marine pollution",
        "fisheries management, microplastics in oceans and coastal conservation"
    ],
    "SDG 15: Life on Land": [
        "biodiversity conservation, forestry, terrestrial ecosystems, soil degradation, wildlife",
        "endangered species, deforestation prevention, ecology and land restoration"
    ],
    "SDG 16: Peace, Justice and Strong Institutions": [
        "cybersecurity, digital forensics, law, governance, public policy, anti-corruption",
        "human rights, privacy protection, blockchain security and institutional transparency"
    ],
    "SDG 17: Partnerships for the Goals": [
        "international partnerships, global collaboration, research collaboration, cross-border cooperation",
        "public private partnerships, technology transfer, capacity building, knowledge sharing, development cooperation",
        "multi-stakeholder partnerships, institutional collaboration, international research networks and shared innovation"
    ]
}

sdg_names = list(sdg_definitions.keys())
sdg_short_labels = [s.split(":")[0] for s in sdg_names]

# Official UN SDG Color Palette
sdg_colors = [
    "#E5243B", "#DDA63A", "#4C9F38", "#C5192D", "#FF3A21", "#26BDE2",
    "#FCC30B", "#A21942", "#FD6925", "#DD1367", "#FD9D24", "#BF8B2E",
    "#3F7E44", "#0A97D9", "#56C02B", "#00689D", "#19486A"
]

# 2. Direct AI/NLP Title-Level SDG Prediction for CU using SentenceTransformer
print("Encoding SDG descriptions and mapping CU publications to UN SDGs...")
sdg_embeddings = {}
for sdg, desc_list in sdg_definitions.items():
    sdg_embeddings[sdg] = model.encode(desc_list, normalize_embeddings=True, show_progress_bar=False)

# Reuse title_embeddings if already in memory, otherwise encode valid_titles
if "title_embeddings" not in globals():
    title_embeddings = model.encode(valid_titles, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

sdg_score_cols = []
for sdg in sdg_names:
    scores = title_embeddings @ sdg_embeddings[sdg].T
    sdg_score_cols.append(scores.max(axis=1))

sdg_score_matrix = np.column_stack(sdg_score_cols)
best_sdg_idx = sdg_score_matrix.argmax(axis=1)
predicted_sdgs = [sdg_names[i] for i in best_sdg_idx]
raw_sdg_scores = sdg_score_matrix[np.arange(len(best_sdg_idx)), best_sdg_idx]

final_predictions["Predicted SDG"] = pd.NA
final_predictions["SDG Alignment Index"] = np.nan
final_predictions.loc[valid_mask, "Predicted SDG"] = predicted_sdgs
final_predictions.loc[valid_mask, "SDG Alignment Index"] = [
    assign_relatedness_index(s) for s in raw_sdg_scores
]

# Save updated CU predictions with SDG column
final_predictions.to_csv("CU_final_research_area_and_SDG_predictions.csv", index=False)

# 3. Cross-Group SDG Mapping Matrix (20 Research Areas -> 17 SDGs)
# SDG 17 is collaboration-specific and cannot be inferred from broad subject-area counts alone.
# It is classified from publication titles; group-level SDG 17 portfolio shares remain unavailable.
area_to_sdg_weights = {
    "Medicine": {"SDG 3: Good Health and Well-Being": 0.85, "SDG 5: Gender Equality": 0.05, "SDG 10: Reduced Inequalities": 0.10},
    "Physics and Astronomy": {"SDG 9: Industry, Innovation and Infrastructure": 0.55, "SDG 7: Affordable and Clean Energy": 0.30, "SDG 13: Climate Action": 0.15},
    "Engineering": {"SDG 9: Industry, Innovation and Infrastructure": 0.50, "SDG 11: Sustainable Cities and Communities": 0.25, "SDG 7: Affordable and Clean Energy": 0.15, "SDG 12: Responsible Consumption and Production": 0.10},
    "Biochemistry, Genetics and Molecular Biology": {"SDG 3: Good Health and Well-Being": 0.65, "SDG 2: Zero Hunger": 0.20, "SDG 15: Life on Land": 0.15},
    "Computer Science": {"SDG 9: Industry, Innovation and Infrastructure": 0.50, "SDG 11: Sustainable Cities and Communities": 0.15, "SDG 4: Quality Education": 0.15, "SDG 16: Peace, Justice and Strong Institutions": 0.10, "SDG 8: Decent Work and Economic Growth": 0.10},
    "Materials Science": {"SDG 9: Industry, Innovation and Infrastructure": 0.40, "SDG 12: Responsible Consumption and Production": 0.35, "SDG 7: Affordable and Clean Energy": 0.25},
    "Chemistry": {"SDG 12: Responsible Consumption and Production": 0.35, "SDG 6: Clean Water and Sanitation": 0.30, "SDG 3: Good Health and Well-Being": 0.20, "SDG 7: Affordable and Clean Energy": 0.15},
    "Social Sciences": {"SDG 4: Quality Education": 0.25, "SDG 8: Decent Work and Economic Growth": 0.20, "SDG 10: Reduced Inequalities": 0.20, "SDG 16: Peace, Justice and Strong Institutions": 0.15, "SDG 5: Gender Equality": 0.10, "SDG 1: No Poverty": 0.10},
    "Mathematics": {"SDG 9: Industry, Innovation and Infrastructure": 0.45, "SDG 4: Quality Education": 0.30, "SDG 8: Decent Work and Economic Growth": 0.25},
    "Earth and Planetary Sciences": {"SDG 13: Climate Action": 0.45, "SDG 15: Life on Land": 0.25, "SDG 14: Life Below Water": 0.15, "SDG 6: Clean Water and Sanitation": 0.15},
    "Agricultural and Biological Sciences": {"SDG 2: Zero Hunger": 0.55, "SDG 15: Life on Land": 0.30, "SDG 14: Life Below Water": 0.15},
    "Environmental Science": {"SDG 13: Climate Action": 0.30, "SDG 6: Clean Water and Sanitation": 0.25, "SDG 15: Life on Land": 0.20, "SDG 12: Responsible Consumption and Production": 0.15, "SDG 14: Life Below Water": 0.10},
    "Chemical Engineering": {"SDG 12: Responsible Consumption and Production": 0.35, "SDG 6: Clean Water and Sanitation": 0.30, "SDG 7: Affordable and Clean Energy": 0.20, "SDG 9: Industry, Innovation and Infrastructure": 0.15},
    "Arts and Humanities": {"SDG 4: Quality Education": 0.40, "SDG 11: Sustainable Cities and Communities": 0.20, "SDG 16: Peace, Justice and Strong Institutions": 0.20, "SDG 5: Gender Equality": 0.20},
    "Multidisciplinary": {"SDG 9: Industry, Innovation and Infrastructure": 0.25, "SDG 3: Good Health and Well-Being": 0.25, "SDG 13: Climate Action": 0.25, "SDG 11: Sustainable Cities and Communities": 0.25},
    "Neuroscience": {"SDG 3: Good Health and Well-Being": 0.85, "SDG 4: Quality Education": 0.15},
    "Immunology and Microbiology": {"SDG 3: Good Health and Well-Being": 0.75, "SDG 6: Clean Water and Sanitation": 0.15, "SDG 2: Zero Hunger": 0.10},
    "Energy": {"SDG 7: Affordable and Clean Energy": 0.70, "SDG 13: Climate Action": 0.20, "SDG 9: Industry, Innovation and Infrastructure": 0.10},
    "Pharmacology, Toxicology and Pharmaceutics": {"SDG 3: Good Health and Well-Being": 0.85, "SDG 12: Responsible Consumption and Production": 0.15},
    "Psychology": {"SDG 3: Good Health and Well-Being": 0.55, "SDG 4: Quality Education": 0.20, "SDG 10: Reduced Inequalities": 0.15, "SDG 5: Gender Equality": 0.10}
}

weight_df = pd.DataFrame(0.0, index=area_names, columns=sdg_names)
for area, s_map in area_to_sdg_weights.items():
    for sdg_k, w_val in s_map.items():
        weight_df.loc[area, sdg_k] = w_val

# Standardized SDG Share (%) for CU, India Top 20, and World Top 20
cu_area_vec = cu_summary.set_index("Research Area").loc[area_names, "CU_Share_%"].values
india_area_vec = india_summary.set_index("Research Area").loc[area_names, "India_Share_%"].values
world_area_vec = world_summary.set_index("Research Area").loc[area_names, "World_Share_%"].values

cu_sdg_mapped_share = (cu_area_vec @ weight_df.values).round(2)
india_sdg_mapped_share = (india_area_vec @ weight_df.values).round(2)
world_sdg_mapped_share = (world_area_vec @ weight_df.values).round(2)

# SDG 17 (Partnerships for the Goals) needs collaboration/network metadata.
# Do not report zero as if it means no partnerships; mark group-level proxy shares unavailable.
sdg17_name = "SDG 17: Partnerships for the Goals"
sdg17_idx = sdg_names.index(sdg17_name)
cu_sdg_mapped_share[sdg17_idx] = np.nan
india_sdg_mapped_share[sdg17_idx] = np.nan
world_sdg_mapped_share[sdg17_idx] = np.nan

# Direct Title-Level AI SDG counts & share for CU
cu_direct_sdg_counts = (
    final_predictions.loc[valid_mask, "Predicted SDG"]
    .value_counts()
    .reindex(sdg_names, fill_value=0)
)
cu_direct_sdg_share = (cu_direct_sdg_counts / cu_direct_sdg_counts.sum() * 100).round(2)

sdg_comparison_df = pd.DataFrame({
    "SDG Goal": sdg_names,
    "SDG_Code": sdg_short_labels,
    "CU_Direct_AI_Publications": cu_direct_sdg_counts.values,
    "CU_Direct_AI_Share_%": cu_direct_sdg_share.values,
    "CU_Portfolio_SDG_Share_%": cu_sdg_mapped_share,
    "India_Top20_SDG_Share_%": india_sdg_mapped_share,
    "World_Top20_SDG_Share_%": world_sdg_mapped_share
})

sdg_comparison_df["Gap_CU_vs_World_SDG_%"] = (
    sdg_comparison_df["CU_Portfolio_SDG_Share_%"] - sdg_comparison_df["World_Top20_SDG_Share_%"]
).round(2)
sdg_comparison_df["Gap_CU_vs_India_SDG_%"] = (
    sdg_comparison_df["CU_Portfolio_SDG_Share_%"] - sdg_comparison_df["India_Top20_SDG_Share_%"]
).round(2)

print("\n✅ UN SDG Alignment Comparison Table (CU vs. India Top 20 vs. World Top 20):")
display(sdg_comparison_df)

# --- GRAPH SDG 1: CU Direct AI-Predicted SDG Distribution (Official UN SDG Colors) ---
fig, ax = plt.subplots(figsize=(13, 8))
y_pos_sdg = np.arange(len(sdg_comparison_df))
bars_sdg1 = ax.barh(
    y_pos_sdg,
    sdg_comparison_df["CU_Direct_AI_Publications"],
    color=sdg_colors,
    edgecolor="black",
    alpha=0.92
)
ax.set_yticks(y_pos_sdg)
ax.set_yticklabels(sdg_comparison_df["SDG Goal"], fontsize=10, weight="bold")
ax.invert_yaxis()
ax.set_xlabel("Number of CU Publications (AI Title Classification)", fontsize=12, weight="bold")
ax.set_title("Chandigarh University Publications Aligned with UN SDG Goals (Direct NLP Classification)", fontsize=14, weight="bold", pad=15)

for i, bar in enumerate(bars_sdg1):
    cnt = int(bar.get_width())
    pct = sdg_comparison_df["CU_Direct_AI_Share_%"].iloc[i]
    ax.text(cnt + max(sdg_comparison_df["CU_Direct_AI_Publications"])*0.01, bar.get_y() + bar.get_height()/2, f"{cnt} ({pct:.1f}%)", va="center", fontsize=9, weight="bold")

ax.set_xlim(0, max(sdg_comparison_df["CU_Direct_AI_Publications"]) * 1.15)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_SDG_DIR, "SDG1_CU_Direct_AI_SDG_Distribution.png"), bbox_inches="tight")
plt.show()

# --- GRAPH SDG 2: 3-Way SDG Alignment Grouped Bar Chart (CU vs India vs World) ---
fig, ax = plt.subplots(figsize=(15, 10))
h_s = 0.27
ax.barh(y_pos_sdg - h_s, sdg_comparison_df["CU_Portfolio_SDG_Share_%"], height=h_s, label="Chandigarh University (CU %)", color="#E84393", edgecolor="black")
ax.barh(y_pos_sdg, sdg_comparison_df["India_Top20_SDG_Share_%"], height=h_s, label="India Top 20 (%)", color="#FDCB6E", edgecolor="black")
ax.barh(y_pos_sdg + h_s, sdg_comparison_df["World_Top20_SDG_Share_%"], height=h_s, label="World Top 20 (%)", color="#0984E3", edgecolor="black")

ax.set_yticks(y_pos_sdg)
ax.set_yticklabels(sdg_comparison_df["SDG Goal"], fontsize=10, weight="bold")
ax.invert_yaxis()
ax.set_xlabel("SDG Alignment Share (%)", fontsize=12, weight="bold")
ax.set_title("UN SDG Goals Alignment Comparison: CU vs. India Top 20 vs. World Top 20", fontsize=15, weight="bold", pad=15)
ax.legend(fontsize=11, loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_SDG_DIR, "SDG2_3Way_SDG_Comparison_Bar.png"), bbox_inches="tight")
plt.show()

# --- GRAPH SDG 3: 3-Way SDG Radar Chart (CU vs India vs World) ---
angles_sdg = np.linspace(0, 2 * np.pi, len(sdg_short_labels), endpoint=False).tolist()
cu_s_r = sdg_comparison_df["CU_Portfolio_SDG_Share_%"].tolist() + [sdg_comparison_df["CU_Portfolio_SDG_Share_%"].iloc[0]]
in_s_r = sdg_comparison_df["India_Top20_SDG_Share_%"].tolist() + [sdg_comparison_df["India_Top20_SDG_Share_%"].iloc[0]]
wo_s_r = sdg_comparison_df["World_Top20_SDG_Share_%"].tolist() + [sdg_comparison_df["World_Top20_SDG_Share_%"].iloc[0]]
angles_sdg += angles_sdg[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
ax.plot(angles_sdg, cu_s_r, color="#E84393", linewidth=2.5, marker="o", label="CU SDG Share (%)")
ax.fill(angles_sdg, cu_s_r, color="#E84393", alpha=0.20)

ax.plot(angles_sdg, in_s_r, color="#F39C12", linewidth=2.5, marker="^", label="India Top 20 SDG Share (%)")
ax.fill(angles_sdg, in_s_r, color="#F39C12", alpha=0.18)

ax.plot(angles_sdg, wo_s_r, color="#0984E3", linewidth=2.5, marker="s", label="World Top 20 SDG Share (%)")
ax.fill(angles_sdg, wo_s_r, color="#0984E3", alpha=0.18)

ax.set_xticks(angles_sdg[:-1])
ax.set_xticklabels(sdg_short_labels, fontsize=10, weight="bold")
ax.set_title("UN SDG Footprint Radar: CU vs. India Top 20 vs. World Top 20", fontsize=14, weight="bold", pad=25)
ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_SDG_DIR, "SDG3_3Way_SDG_Radar.png"), bbox_inches="tight")
plt.show()

# --- GRAPH SDG 4: Dual Diverging SDG Gap Chart (CU vs World & CU vs India) ---
fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharey=True)

# Panel 1: CU vs World SDG Gap
c_w_sdg = ["#00B894" if v >= 0 else "#D63031" for v in sdg_comparison_df["Gap_CU_vs_World_SDG_%"]]
axes[0].barh(sdg_comparison_df["SDG Goal"], sdg_comparison_df["Gap_CU_vs_World_SDG_%"], color=c_w_sdg, edgecolor="black")
axes[0].axvline(0, color="black", linestyle="--")
axes[0].set_title("CU vs. World Top 20 SDG Gap (%)", fontsize=13, weight="bold")
axes[0].set_xlabel("CU % minus World %")
axes[0].invert_yaxis()

# Panel 2: CU vs India SDG Gap
c_i_sdg = ["#0984E3" if v >= 0 else "#E17055" for v in sdg_comparison_df["Gap_CU_vs_India_SDG_%"]]
axes[1].barh(sdg_comparison_df["SDG Goal"], sdg_comparison_df["Gap_CU_vs_India_SDG_%"], color=c_i_sdg, edgecolor="black")
axes[1].axvline(0, color="black", linestyle="--")
axes[1].set_title("CU vs. India Top 20 SDG Gap (%)", fontsize=13, weight="bold")
axes[1].set_xlabel("CU % minus India %")

plt.suptitle("UN SDG Directional Gap Analysis: Where CU Leads vs. Where Global/National Peers Lead", fontsize=15, weight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_SDG_DIR, "SDG4_Dual_SDG_Gap_Analysis.png"), bbox_inches="tight")
plt.show()

# --- GRAPH SDG 5: Research Area x SDG Cross-Tab Heatmap (Actual CU Publications) ---
cu_area_sdg_crosstab = pd.crosstab(
    final_predictions.loc[valid_mask, "Related To"],
    final_predictions.loc[valid_mask, "Predicted SDG"]
).reindex(index=area_names, columns=sdg_names, fill_value=0)
cu_area_sdg_crosstab.columns = sdg_short_labels

plt.figure(figsize=(15, 9))
sns.heatmap(cu_area_sdg_crosstab, cmap="magma_r", annot=True, fmt="d", linewidths=0.5)
plt.title("CU Research Area to UN SDG Alignment Matrix (Publication Count per Cell)", fontsize=14, weight="bold", pad=15)
plt.xlabel("UN Sustainable Development Goals (SDGs)", fontsize=12, weight="bold")
plt.ylabel("20 Predicted Research Areas", fontsize=12, weight="bold")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_SDG_DIR, "SDG5_CU_Area_to_SDG_Matrix_Heatmap.png"), bbox_inches="tight")
plt.show()

print("✅ All 5 SDG Alignment Charts generated and saved in:", PLOTS_SDG_DIR)


In [ ]:
# ==============================================================================
# STEP 22: EXPORT MASTER EXCEL REPORT & COMPLETE ZIP ARCHIVE
# ==============================================================================

from google.colab import files

master_excel_path = os.path.join(COMP_OUTPUT_DIR, "CU_World_India_SDG_Master_Report.xlsx")

with pd.ExcelWriter(master_excel_path, engine="openpyxl") as writer:
    master_comp_df.to_excel(writer, sheet_name="3Way_Master_Comparison", index=False)
    cu_vs_world_df.to_excel(writer, sheet_name="CU_vs_World_Top20", index=False)
    cu_vs_india_df.to_excel(writer, sheet_name="CU_vs_India_Top20", index=False)
    sdg_comparison_df.to_excel(writer, sheet_name="SDG_Alignment_Comparison", index=False)
    cu_area_sdg_crosstab.reset_index().to_excel(writer, sheet_name="CU_Area_x_SDG_Matrix", index=False)
    world_heatmap_df.reset_index().to_excel(writer, sheet_name="World_20_Uni_Share_Matrix", index=False)
    india_heatmap_df.reset_index().to_excel(writer, sheet_name="India_20_Uni_Share_Matrix", index=False)

comp_zip_path = "/content/CU_Comparison_and_SDG_Complete_Results.zip"
with zipfile.ZipFile(comp_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk(COMP_OUTPUT_DIR):
        for filename in filenames:
            full_p = os.path.join(root, filename)
            arcname = os.path.relpath(full_p, COMP_OUTPUT_DIR)
            zf.write(full_p, arcname)

print("✅ Master Excel Report saved at:", master_excel_path)
print("✅ Complete Plots & Reports ZIP saved at:", comp_zip_path)

files.download(master_excel_path)
files.download(comp_zip_path)


In [ ]:
# ==============================================================================
# STEP 23: CU RESEARCH INTELLIGENCE — PREMIUM INTERACTIVE GRADIO DASHBOARD
# ==============================================================================
%pip install -q gradio openpyxl

import gradio as gr
import tempfile
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os

# A cohesive visual system for all dashboard charts.
DASH_COLORS = {
    "cu": "#7C3AED", "india": "#F59E0B", "world": "#06B6D4",
    "upload": "#EC4899", "positive": "#10B981", "negative": "#EF4444",
    "ink": "#172554", "muted": "#64748B"
}
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 115, "savefig.dpi": 220,
    "axes.titleweight": "bold", "axes.labelweight": "semibold",
    "axes.edgecolor": "#D8DEEA", "grid.alpha": 0.22,
    "font.family": "DejaVu Sans"
})

def _style_ax(ax):
    ax.set_facecolor("#FBFCFF")
    ax.grid(axis="x", alpha=0.22)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_color("#E2E8F0")

def _benchmark_frame(mode):
    if mode == "CU vs. World Top 20":
        return cu_vs_world_df.copy()
    if mode == "CU vs. India Top 20":
        return cu_vs_india_df.copy()
    return master_comp_df.copy()

def render_benchmark_view(comp_mode, chart_type, top_n):
    top_n = int(top_n)
    df_sub = _benchmark_frame(comp_mode)
    if chart_type == "Institution-Level Heatmap":
        if comp_mode == "CU vs. World Top 20":
            heat = world_uni_shares
            title = "World Top 20 — university research portfolio (%)"
        elif comp_mode == "CU vs. India Top 20":
            heat = india_uni_shares
            title = "India Top 20 — university research portfolio (%)"
        else:
            heat = pd.concat([world_uni_shares.add_prefix("World · "),
                              india_uni_shares.add_prefix("India · ")], axis=1)
            title = "40-university research portfolio map (%)"
        fig, ax = plt.subplots(figsize=(17, 9))
        sns.heatmap(heat, cmap="mako", linewidths=.25, linecolor="white",
                    cbar_kws={"label": "Share of each university's mapped publications (%)"},
                    ax=ax)
        ax.set_title(title, pad=16, color=DASH_COLORS["ink"], fontsize=15)
        ax.set_xlabel("Universities")
        ax.set_ylabel("Research areas")
        plt.xticks(rotation=55, ha="right", fontsize=8)
        plt.yticks(fontsize=8)
        plt.tight_layout()
        return fig, df_sub, _benchmark_summary(comp_mode, df_sub), _benchmark_insights(comp_mode, df_sub)

    if chart_type == "Research Share — Grouped Bars":
        p = df_sub.sort_values("CU_Share_%", ascending=True).tail(top_n)
        fig, ax = plt.subplots(figsize=(13, max(6, top_n * .42)))
        y = np.arange(len(p)); h = .24 if comp_mode.startswith("3-Way") else .34
        ax.barh(y + h, p["CU_Share_%"], height=h, color=DASH_COLORS["cu"], label="Chandigarh University")
        if comp_mode == "CU vs. World Top 20":
            ax.barh(y, p["World_Share_%"], height=h, color=DASH_COLORS["world"], label="World Top 20")
        elif comp_mode == "CU vs. India Top 20":
            ax.barh(y, p["India_Share_%"], height=h, color=DASH_COLORS["india"], label="India Top 20")
        else:
            ax.barh(y, p["India_Share_%"], height=h, color=DASH_COLORS["india"], label="India Top 20")
            ax.barh(y - h, p["World_Share_%"], height=h, color=DASH_COLORS["world"], label="World Top 20")
        ax.set_yticks(y); ax.set_yticklabels(p["Research Area"], fontsize=9)
        ax.set_xlabel("Share of mapped publications (%)")
        ax.set_title(f"Research portfolio comparison · {comp_mode}", pad=14)
        ax.legend(ncol=3, frameon=True, loc="lower right")
        _style_ax(ax)

    elif chart_type == "Strategic Gap — CU minus Benchmark":
        fig, ax = plt.subplots(figsize=(13, max(6, top_n * .42)))
        p = df_sub.copy()
        if comp_mode == "CU vs. World Top 20":
            cols = [("Share_Gap_% (CU - World)", "CU − World", DASH_COLORS["world"])]
        elif comp_mode == "CU vs. India Top 20":
            cols = [("Share_Gap_% (CU - India)", "CU − India", DASH_COLORS["india"])]
        else:
            cols = [("Gap_CU_vs_India_%", "CU − India", DASH_COLORS["india"]),
                    ("Gap_CU_vs_World_%", "CU − World", DASH_COLORS["world"])]
        sort_col = cols[0][0]
        p = p.sort_values(sort_col).tail(top_n)
        y = np.arange(len(p))
        if len(cols) == 1:
            vals = p[cols[0][0]].values
            ax.barh(y, vals, color=[DASH_COLORS["positive"] if v >= 0 else DASH_COLORS["negative"] for v in vals])
        else:
            h = .34
            for offset, (col, label, color) in zip([h/2, -h/2], cols):
                ax.barh(y + offset, p[col], height=h, color=color, label=label)
            ax.legend()
        ax.set_yticks(y); ax.set_yticklabels(p["Research Area"], fontsize=9)
        ax.axvline(0, color="#334155", linestyle="--", linewidth=1.2)
        ax.set_xlabel("Percentage-point gap (positive = CU has a higher share)")
        ax.set_title("Where CU is ahead — and where peers are ahead", pad=14)
        _style_ax(ax)

    elif chart_type == "Research Profile — Radar":
        p = df_sub.sort_values("CU_Share_%", ascending=False).head(min(top_n, 10))
        labels = p["Research Area"].tolist()
        angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
        angles += angles[:1]
        fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={"polar": True})
        series = [("CU_Share_%", "Chandigarh University", DASH_COLORS["cu"])]
        if comp_mode in ["CU vs. India Top 20", "3-Way Master (CU vs. India vs. World)"]:
            series.append(("India_Share_%", "India Top 20", DASH_COLORS["india"]))
        if comp_mode in ["CU vs. World Top 20", "3-Way Master (CU vs. India vs. World)"]:
            series.append(("World_Share_%", "World Top 20", DASH_COLORS["world"]))
        for col, label, color in series:
            vals = p[col].tolist(); vals += vals[:1]
            ax.plot(angles, vals, color=color, linewidth=2.5, marker="o", label=label)
            ax.fill(angles, vals, color=color, alpha=.10)
        ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels, fontsize=8)
        ax.set_title("Research portfolio fingerprint · top CU focus areas", pad=24)
        ax.legend(loc="upper right", bbox_to_anchor=(1.32, 1.12))

    elif chart_type == "Priority Rank Comparison":
        p = df_sub.sort_values("CU_Rank", ascending=False).tail(top_n)
        fig, ax = plt.subplots(figsize=(12, max(6, top_n * .4)))
        y = np.arange(len(p))
        ax.scatter(p["CU_Rank"], y, s=75, color=DASH_COLORS["cu"], label="CU rank", zorder=3)
        if comp_mode == "CU vs. World Top 20":
            ax.scatter(p["World_Rank"], y, s=75, color=DASH_COLORS["world"], label="World rank", zorder=3)
            ax.hlines(y, p["CU_Rank"], p["World_Rank"], color="#CBD5E1", linewidth=2)
        elif comp_mode == "CU vs. India Top 20":
            ax.scatter(p["India_Rank"], y, s=75, color=DASH_COLORS["india"], label="India rank", zorder=3)
            ax.hlines(y, p["CU_Rank"], p["India_Rank"], color="#CBD5E1", linewidth=2)
        else:
            ax.scatter(p["India_Rank"], y, s=65, color=DASH_COLORS["india"], label="India rank", zorder=3)
            ax.scatter(p["World_Rank"], y, s=65, color=DASH_COLORS["world"], label="World rank", zorder=3)
            ax.hlines(y, p[["CU_Rank","India_Rank","World_Rank"]].min(axis=1),
                      p[["CU_Rank","India_Rank","World_Rank"]].max(axis=1), color="#CBD5E1", linewidth=2)
        ax.set_yticks(y); ax.set_yticklabels(p["Research Area"], fontsize=9)
        ax.invert_xaxis(); ax.set_xlabel("Priority rank (1 = highest publication volume)")
        ax.set_title("Research priority rank — lower number means higher output")
        ax.legend(ncol=3)
        _style_ax(ax)

    else:  # Strategic positioning bubble chart
        p = df_sub.copy()
        fig, ax = plt.subplots(figsize=(12, 8))
        sizes = p["CU_Share_%"].clip(lower=0) * 45 + 65
        sc = ax.scatter(p["World_Share_%"], p["India_Share_%"], s=sizes,
                        c=p["CU_Share_%"], cmap="plasma", alpha=.82,
                        edgecolors="white", linewidth=1.2)
        lim = max(p["World_Share_%"].max(), p["India_Share_%"].max()) * 1.12
        ax.plot([0, lim], [0, lim], "--", color="#64748B", label="Equal World/India share")
        for _, row in p.nlargest(min(top_n, 12), "CU_Share_%").iterrows():
            ax.annotate(row["Research Area"], (row["World_Share_%"], row["India_Share_%"]),
                        xytext=(5, 5), textcoords="offset points", fontsize=7)
        ax.set_xlim(0, lim); ax.set_ylim(0, lim)
        ax.set_xlabel("World Top 20 share (%)"); ax.set_ylabel("India Top 20 share (%)")
        ax.set_title("Strategic positioning map · bubble size and colour represent CU share")
        fig.colorbar(sc, ax=ax, label="CU share (%)")
        ax.legend()
        _style_ax(ax)

    plt.tight_layout()
    return fig, df_sub, _benchmark_summary(comp_mode, df_sub), _benchmark_insights(comp_mode, df_sub)

def _benchmark_summary(mode, frame):
    top = frame.sort_values("CU_Share_%", ascending=False).head(3)
    return (
        f"### Portfolio snapshot · {mode}\n"
        f"- **Research areas analysed:** {len(frame)}\n"
        f"- **CU's leading focus:** {top.iloc[0]['Research Area']} "
        f"({top.iloc[0]['CU_Share_%']:.2f}% of mapped CU publications)\n"
        f"- **Next focus areas:** " + ", ".join(top.iloc[1:]["Research Area"].tolist())
    )

def _benchmark_insights(mode, frame):
    if mode == "CU vs. World Top 20":
        col = "Share_Gap_% (CU - World)"
        label = "World"
    elif mode == "CU vs. India Top 20":
        col = "Share_Gap_% (CU - India)"
        label = "India"
    else:
        row_i = frame.loc[frame["Gap_CU_vs_India_%"].abs().idxmax()]
        row_w = frame.loc[frame["Gap_CU_vs_World_%"].abs().idxmax()]
        return (
            "### Signals to explore\n"
            f"- **Largest CU–India difference:** {row_i['Research Area']} "
            f"({row_i['Gap_CU_vs_India_%']:+.2f} percentage points)\n"
            f"- **Largest CU–World difference:** {row_w['Research Area']} "
            f"({row_w['Gap_CU_vs_World_%']:+.2f} percentage points)\n"
            "- A positive gap means CU has a larger share in that area; a negative gap means the benchmark does."
        )
    high = frame.loc[frame[col].idxmax()]
    low = frame.loc[frame[col].idxmin()]
    close_n = int((frame[col].abs() <= 1).sum())
    return (
        "### Signals to explore\n"
        f"- **CU's strongest relative focus vs. {label}:** {high['Research Area']} ({high[col]:+.2f} pp)\n"
        f"- **Benchmark's strongest relative focus:** {low['Research Area']} ({low[col]:+.2f} pp)\n"
        f"- **Closely aligned areas (within ±1 pp):** {close_n} of {len(frame)}\n"
        "- Percentage-point gaps compare portfolio shares, not raw publication counts."
    )

def render_university_comparison(selection, chart_mode):
    """Compare CU's research portfolio with one selected built-in university."""
    if not selection:
        raise gr.Error("Choose a university to compare with CU.")

    group, uni = selection.split("::", 1)
    matrix = world_uni_shares if group == "WORLD" else india_uni_shares
    counts = world_uni_counts if group == "WORLD" else india_uni_counts

    if uni not in matrix.columns:
        raise gr.Error("University data could not be found.")

    cu_counts = cu_summary.set_index("Research Area")["CU_Publications"].reindex(area_names).fillna(0)
    cu_shares = cu_summary.set_index("Research Area")["CU_Share_%"].reindex(area_names).fillna(0)
    uni_counts = counts[uni].reindex(area_names).fillna(0)
    uni_shares = matrix[uni].reindex(area_names).fillna(0)

    comparison = pd.DataFrame({
        "Research Area": area_names,
        "CU Publications": cu_counts.values.astype(int),
        "CU Share (%)": cu_shares.values,
        "CU Rank": cu_shares.rank(ascending=False, method="min").astype(int).values,
        f"{uni} Publications": uni_counts.values.astype(int),
        f"{uni} Share (%)": uni_shares.values,
        f"{uni} Rank": uni_shares.rank(ascending=False, method="min").astype(int).values,
        "Share Gap (CU - Selected, pp)": (cu_shares.values - uni_shares.values).round(2)
    })

    if chart_mode == "Research portfolio comparison":
        focus = comparison.assign(_priority=comparison[["CU Share (%)", f"{uni} Share (%)"]].max(axis=1))
        plot_df = focus.nlargest(12, "_priority").sort_values("_priority")
        fig, ax = plt.subplots(figsize=(12, 7))
        y = np.arange(len(plot_df))
        h = .34
        ax.barh(y + h/2, plot_df["CU Share (%)"], height=h,
                color=DASH_COLORS["cu"], label="Chandigarh University")
        ax.barh(y - h/2, plot_df[f"{uni} Share (%)"], height=h,
                color=DASH_COLORS["world"] if group == "WORLD" else DASH_COLORS["india"],
                label=uni)
        ax.set_yticks(y)
        ax.set_yticklabels(plot_df["Research Area"], fontsize=9)
        ax.set_xlabel("Share of mapped publications (%)")
        ax.set_title(f"CU vs. {uni} · Research portfolio", pad=14)
        ax.legend()
        _style_ax(ax)
    else:
        plot_df = comparison.reindex(
            comparison["Share Gap (CU - Selected, pp)"].abs().sort_values(ascending=False).index
        ).head(12).sort_values("Share Gap (CU - Selected, pp)")
        fig, ax = plt.subplots(figsize=(12, 7))
        gap = plot_df["Share Gap (CU - Selected, pp)"].values
        ax.barh(plot_df["Research Area"], gap,
                color=[DASH_COLORS["positive"] if v >= 0 else DASH_COLORS["negative"] for v in gap])
        ax.axvline(0, color="#334155", linestyle="--", linewidth=1.2)
        ax.set_xlabel("Percentage-point difference (positive = higher CU share)")
        ax.set_title(f"Research focus differences · CU vs. {uni}", pad=14)
        _style_ax(ax)

    comparison = comparison.sort_values("CU Share (%)", ascending=False).reset_index(drop=True)
    cu_top = comparison.loc[comparison["CU Share (%)"].idxmax()]
    uni_top = comparison.loc[comparison[f"{uni} Share (%)"].idxmax()]
    mean_gap = comparison["Share Gap (CU - Selected, pp)"].abs().mean()
    group_label = "World Top 20" if group == "WORLD" else "India Top 20"
    note = (
        f"### CU × {uni}\n"
        f"- **Benchmark cohort:** {group_label}\n"
        f"- **CU leading area:** {cu_top['Research Area']} ({cu_top['CU Share (%)']:.2f}%)\n"
        f"- **Selected university leading area:** {uni_top['Research Area']} ({uni_top[f'{uni} Share (%)']:.2f}%)\n"
        f"- **Average absolute portfolio-share difference:** {mean_gap:.2f} percentage points\n\n"
        "Shares are calculated within the same 20 mapped research areas; they compare portfolio emphasis, "
        "not institutional quality or overall university rankings."
    )
    return fig, comparison, note
def render_sdg_view(sdg_chart_mode):
    if sdg_chart_mode == "CU Direct AI-Predicted SDG Bar Chart":
        fig, ax = plt.subplots(figsize=(13, 7))
        p = sdg_comparison_df.sort_values("CU_Direct_AI_Publications", ascending=True)
        ax.barh(p["SDG Goal"], p["CU_Direct_AI_Publications"], color=sdg_colors)
        ax.set_xlabel("CU publications classified into each SDG")
        ax.set_title("CU publication alignment · direct AI title classification")
    elif sdg_chart_mode == "3-Way SDG Share Comparison":
        fig, ax = plt.subplots(figsize=(13, 8))
        p = sdg_comparison_df
        y = np.arange(len(p)); h = .25
        ax.barh(y+h, p["CU_Portfolio_SDG_Share_%"], height=h, color=DASH_COLORS["cu"], label="CU")
        ax.barh(y, p["India_Top20_SDG_Share_%"], height=h, color=DASH_COLORS["india"], label="India Top 20")
        ax.barh(y-h, p["World_Top20_SDG_Share_%"], height=h, color=DASH_COLORS["world"], label="World Top 20")
        ax.set_yticks(y); ax.set_yticklabels(p["SDG Goal"], fontsize=8)
        ax.set_xlabel("Estimated SDG alignment share (%)")
        ax.set_title("UN SDG portfolio comparison · CU vs. India vs. World")
        ax.legend(ncol=3)
    elif sdg_chart_mode == "3-Way SDG Radar":
        fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={"polar": True})
        angles = np.linspace(0, 2*np.pi, len(sdg_short_labels), endpoint=False).tolist()
        angles += angles[:1]
        for col, label, color in [
            ("CU_Portfolio_SDG_Share_%", "CU", DASH_COLORS["cu"]),
            ("India_Top20_SDG_Share_%", "India Top 20", DASH_COLORS["india"]),
            ("World_Top20_SDG_Share_%", "World Top 20", DASH_COLORS["world"])
        ]:
            vals = sdg_comparison_df[col].tolist(); vals += vals[:1]
            ax.plot(angles, vals, color=color, linewidth=2, marker="o", label=label)
            ax.fill(angles, vals, color=color, alpha=.08)
        ax.set_xticks(angles[:-1]); ax.set_xticklabels(sdg_short_labels, fontsize=8)
        ax.set_title("SDG alignment fingerprint", pad=22); ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))
    else:
        fig, ax = plt.subplots(figsize=(14, 8))
        sns.heatmap(cu_area_sdg_crosstab, cmap="rocket", annot=True, fmt="d",
                    linewidths=.25, linecolor="white", ax=ax)
        ax.set_title("CU research area × UN SDG matrix")
        ax.set_xlabel("Sustainable Development Goals"); ax.set_ylabel("Research area")
    _style_ax(ax); plt.tight_layout()
    return fig, sdg_comparison_df

def analyze_custom_uploaded_file(uploaded_file, benchmark_choice):
    if uploaded_file is None:
        raise gr.Error("Upload one Excel or CSV file to begin.")
    fpath = uploaded_file if isinstance(uploaded_file, str) else getattr(uploaded_file, "name", str(uploaded_file))
    if fpath.lower().endswith((".xlsx", ".xls")):
        user_df = pd.read_excel(fpath)
    else:
        try:
            user_df = pd.read_csv(fpath, encoding="utf-8-sig", low_memory=False)
        except UnicodeDecodeError:
            user_df = pd.read_csv(fpath, encoding="latin-1", low_memory=False)

    user_df.columns = [str(c).strip() for c in user_df.columns]
    if user_df.empty:
        raise gr.Error("The uploaded file contains no data rows.")

    total_r, total_c = user_df.shape
    missing_cells = int(user_df.isna().sum().sum())
    dup_rows = int(user_df.duplicated().sum())
    lower_cols = {c.lower(): c for c in user_df.columns}

    if "documentscount" in lower_cols and "title" in lower_cols:
        mode_detected = "University subject-area summary (documentsCount + title)"
        t_col, c_col = lower_cols["title"], lower_cols["documentscount"]
        user_df[c_col] = pd.to_numeric(user_df[c_col], errors="coerce").fillna(0)
        user_df["Mapped_Area"] = user_df[t_col].apply(map_to_20_areas)
        aligned = user_df[user_df["Mapped_Area"].notna()]
        area_series = aligned.groupby("Mapped_Area")[c_col].sum().reindex(area_names, fill_value=0)
        u_share_vec = (area_series / max(area_series.sum(), 1) * 100).values
        sdg_values = (u_share_vec @ weight_df.values).round(2)
        sdg_series = pd.Series(sdg_values, index=sdg_names)
        sdg_ylabel = "Estimated SDG alignment share (%)"
    else:
        mode_detected = "Raw publication dataset (AI title classification)"
        t_col = None
        for cand in ["publication title", "paper title", "document title", "article title", "title", "name"]:
            t_col = next((c for c in user_df.columns if cand in c.lower()), None)
            if t_col:
                break
        if t_col is None:
            t_col = user_df.columns[0]
        titles_clean = user_df[t_col].fillna("").astype(str).str.strip()
        valid_mask = titles_clean.ne("")
        title_list = titles_clean[valid_mask].tolist()
        if not title_list:
            raise gr.Error("No non-empty publication titles were found in the uploaded file.")
        emb = model.encode(title_list, batch_size=64, show_progress_bar=False, normalize_embeddings=True)
        area_scores = np.column_stack([(emb @ variant_embeddings[a].T).max(axis=1) for a in area_names])
        pred_area = [area_names[i] for i in area_scores.argmax(axis=1)]
        sdg_scores = np.column_stack([(emb @ sdg_embeddings[s].T).max(axis=1) for s in sdg_names])
        pred_sdg = [sdg_names[i] for i in sdg_scores.argmax(axis=1)]
        user_df["Predicted Research Area"] = pd.NA
        user_df.loc[valid_mask, "Predicted Research Area"] = pred_area
        user_df["Predicted SDG"] = pd.NA
        user_df.loc[valid_mask, "Predicted SDG"] = pred_sdg
        area_series = user_df.loc[valid_mask, "Predicted Research Area"].value_counts().reindex(area_names, fill_value=0)
        sdg_series = user_df.loc[valid_mask, "Predicted SDG"].value_counts().reindex(sdg_names, fill_value=0)
        sdg_ylabel = "Number of AI-classified publications"

    total_mapped = float(area_series.sum())
    if total_mapped <= 0:
        raise gr.Error("No records could be mapped to the 20 research areas.")
    comp_out = pd.DataFrame({
        "Research Area": area_names,
        "Uploaded_Publications": area_series.values,
        "Uploaded_Share_%": (area_series.values / total_mapped * 100).round(2),
        "CU_Share_%": cu_summary.set_index("Research Area").loc[area_names, "CU_Share_%"].values,
        "India_Top20_Share_%": india_summary.set_index("Research Area").loc[area_names, "India_Share_%"].values,
        "World_Top20_Share_%": world_summary.set_index("Research Area").loc[area_names, "World_Share_%"].values
    }).sort_values("Uploaded_Share_%", ascending=False).reset_index(drop=True)

    fig1, ax1 = plt.subplots(figsize=(13, 7))
    p = comp_out.head(15).sort_values("Uploaded_Share_%", ascending=True)
    y = np.arange(len(p)); h = .25
    ax1.barh(y+h, p["Uploaded_Share_%"], height=h, color=DASH_COLORS["upload"], label="Uploaded dataset")
    if benchmark_choice in ["Compare with World Top 20", "Compare with All (CU + India + World)"]:
        ax1.barh(y, p["World_Top20_Share_%"], height=h, color=DASH_COLORS["world"], label="World Top 20")
    if benchmark_choice in ["Compare with India Top 20", "Compare with All (CU + India + World)"]:
        ax1.barh(y-h, p["India_Top20_Share_%"], height=h, color=DASH_COLORS["india"], label="India Top 20")
    if benchmark_choice == "Compare with All (CU + India + World)":
        ax1.barh(y+2*h, p["CU_Share_%"], height=h, color=DASH_COLORS["cu"], label="CU")
    ax1.set_yticks(y); ax1.set_yticklabels(p["Research Area"], fontsize=9)
    ax1.set_xlabel("Share of publications (%)"); ax1.set_title("Your dataset vs. built-in university benchmarks")
    ax1.legend(ncol=2); _style_ax(ax1); plt.tight_layout()

    fig2, ax2 = plt.subplots(figsize=(13, 7))
    sdg_p = sdg_series.sort_values(ascending=True)
    ax2.barh(sdg_p.index, sdg_p.values, color=sns.color_palette("husl", len(sdg_p)))
    ax2.set_xlabel(sdg_ylabel); ax2.set_title("Uploaded dataset · UN SDG alignment")
    _style_ax(ax2); plt.tight_layout()

    tmp_excel = os.path.join(tempfile.gettempdir(), "CU_Live_Research_Intelligence_Report.xlsx")
    with pd.ExcelWriter(tmp_excel, engine="openpyxl") as writer:
        comp_out.to_excel(writer, sheet_name="Benchmark_Comparison", index=False)
        pd.DataFrame({"SDG Goal": sdg_series.index, "Uploaded SDG Value": sdg_series.values}).to_excel(writer, sheet_name="SDG_Analysis", index=False)
        user_df.head(10000).to_excel(writer, sheet_name="Analysed_Records", index=False)

    top = comp_out.iloc[0]
    summary_md = (
        f"### Analysis complete · {mode_detected}\n"
        f"- **Records:** {total_r:,} · **Columns:** {total_c}\n"
        f"- **Missing cells:** {missing_cells:,} · **Duplicate rows:** {dup_rows:,}\n"
        f"- **Top research area:** {top['Research Area']} ({top['Uploaded_Share_%']:.2f}%)\n"
        f"- **Benchmark mode:** {benchmark_choice}\n"
        "- The 40 comparison university datasets were loaded automatically from this notebook."
    )
    return summary_md, fig1, fig2, comp_out, tmp_excel

# Build a university selector from the embedded benchmark matrices.
university_choices = (
    [f"WORLD::{u}" for u in world_uni_shares.columns] +
    [f"INDIA::{u}" for u in india_uni_shares.columns]
)
overview_text = (
    "## Research Intelligence at a glance\n"
    f"**40 built-in benchmark institutions** · **20 World + 20 India** · "
    f"**{len(area_names)} research areas** · **{len(sdg_names)} UN SDGs**\n\n"
    "Use the tabs to explore portfolio gaps, institution-level fingerprints, SDG alignment, "
    "or upload one dataset for an instant comparison against all built-in benchmarks."
)

custom_css = """
.gradio-container {max-width: 1500px !important; background: #f5f7fc !important;}
#hero {padding: 30px 34px; border-radius: 22px; color: white;
background: radial-gradient(circle at 88% 15%,rgba(255,255,255,.22),transparent 24%),
linear-gradient(115deg,#1e1b4b 0%,#5b21b6 48%,#0e7490 100%);
box-shadow: 0 16px 38px rgba(49,46,129,.20); margin-bottom: 18px;}
#hero h1 {font-size: 32px !important; letter-spacing: -.6px; margin-bottom: 8px;}
#hero p {font-size: 15px; opacity: .96; line-height: 1.6;}
.section-card {border: 1px solid #e4e9f3; border-radius: 18px; background: white; padding: 16px;}
button.primary {background: linear-gradient(90deg,#6d28d9,#0891b2) !important; border: 0 !important; border-radius: 12px !important;}
button {border-radius: 11px !important;}
.tabs {gap: 8px;}
.tab-nav button {font-weight: 650 !important; border-radius: 12px 12px 0 0 !important;}
.block, .form {border-radius: 16px !important;}
.gradio-container {padding-bottom: 30px !important;}
footer {visibility: hidden;}
"""

with gr.Blocks(theme=gr.themes.Soft(), css=custom_css,
               title="CU Research Intelligence & Benchmarking") as demo:
    gr.HTML("""
    <div id="hero">
      <h1>🎓 CU Research Intelligence</h1>
      <p>Research portfolio analytics · Global & Indian benchmarking · AI research mapping · UN SDG alignment</p>
      <p><b>Chandigarh University</b> compared with 20 global and 20 Indian institutions</p>
    </div>
    """)
    gr.Markdown(overview_text)

    with gr.Tabs():
        with gr.TabItem("📊 Benchmark Studio"):
            with gr.Row():
                comp_dropdown = gr.Dropdown(
                    choices=["CU vs. World Top 20", "CU vs. India Top 20",
                             "3-Way Master (CU vs. India vs. World)"],
                    value="3-Way Master (CU vs. India vs. World)",
                    label="Benchmark cohort", scale=2)
                chart_dropdown = gr.Dropdown(
                    choices=["Research Share — Grouped Bars",
                             "Strategic Gap — CU minus Benchmark",
                             "Research Profile — Radar",
                             "Priority Rank Comparison",
                             "Strategic Positioning Bubble Map",
                             "Institution-Level Heatmap"],
                    value="Research Share — Grouped Bars",
                    label="Visualization", scale=2)
                topn_slider = gr.Slider(5, 20, value=15, step=1,
                                        label="Research areas to display", scale=1)
            run_btn1 = gr.Button("✨ Generate benchmark view", variant="primary")
            with gr.Row():
                summary_out = gr.Markdown(elem_classes=["section-card"])
                insights_out = gr.Markdown(elem_classes=["section-card"])
            plot_out1 = gr.Plot(label="Interactive research comparison")
            table_out1 = gr.Dataframe(label="Detailed comparison data", interactive=False)
            run_btn1.click(render_benchmark_view,
                inputs=[comp_dropdown, chart_dropdown, topn_slider],
                outputs=[plot_out1, table_out1, summary_out, insights_out])
            demo.load(render_benchmark_view,
                inputs=[comp_dropdown, chart_dropdown, topn_slider],
                outputs=[plot_out1, table_out1, summary_out, insights_out])

        with gr.TabItem("🏛️ University Explorer"):
            gr.Markdown(
                "### Compare Chandigarh University with any one of the 40 built-in universities\n"
                "Choose an Indian or global institution to compare research-area shares, publication volumes, "
                "priority ranks and portfolio gaps. No additional file upload is needed."
            )
            with gr.Row():
                uni_select = gr.Dropdown(
                    choices=university_choices,
                    label="Select a university",
                    value=university_choices[0] if university_choices else None,
                    filterable=True,
                    scale=2
                )
                uni_chart_mode = gr.Radio(
                    choices=["Research portfolio comparison", "Research focus gap"],
                    value="Research portfolio comparison",
                    label="Analysis view",
                    scale=2
                )
            uni_btn = gr.Button("⚖️ Compare with CU", variant="primary")
            uni_note = gr.Markdown()
            uni_plot = gr.Plot(label="CU vs selected university")
            uni_table = gr.Dataframe(
                label="Detailed research-area comparison",
                interactive=False
            )
            uni_btn.click(
                render_university_comparison,
                inputs=[uni_select, uni_chart_mode],
                outputs=[uni_plot, uni_table, uni_note]
            )
            demo.load(
                render_university_comparison,
                inputs=[uni_select, uni_chart_mode],
                outputs=[uni_plot, uni_table, uni_note]
            )

        with gr.TabItem("🎯 SDG Alignment"):
            gr.Markdown(
                "Explore all **17 UN Sustainable Development Goals**. SDG 17 is classified from publication-title "
                "language where partnership themes are explicit. The 20-area university summary files do not "
                "contain collaboration-network fields, so SDG 17 group-level portfolio shares are shown as "
                "not available rather than incorrectly treating them as zero."
            )
            sdg_mode_dropdown = gr.Dropdown(
                choices=["3-Way SDG Share Comparison", "3-Way SDG Radar",
                         "CU Direct AI-Predicted SDG Bar Chart",
                         "CU Research Area × SDG Heatmap"],
                value="3-Way SDG Share Comparison", label="SDG visualization")
            run_btn2 = gr.Button("🌱 Explore SDG alignment", variant="primary")
            plot_out2 = gr.Plot(label="UN Sustainable Development Goals")
            table_out2 = gr.Dataframe(label="SDG comparison data", interactive=False)
            run_btn2.click(render_sdg_view, inputs=sdg_mode_dropdown,
                           outputs=[plot_out2, table_out2])
            demo.load(render_sdg_view, inputs=sdg_mode_dropdown,
                      outputs=[plot_out2, table_out2])

        with gr.TabItem("📂 Analyse Your Dataset"):
            gr.Markdown(
                "Upload **one** Excel or CSV dataset. The dashboard will run data checks, "
                "classify publication titles when needed, map research areas and SDGs, "
                "and compare the result with the 40 preloaded benchmark institutions.\n\n"
                "**The 40 World/India university files are already built in — do not upload them.**"
            )
            with gr.Row():
                file_input = gr.File(label="Upload one Excel (.xlsx/.xls) or CSV file",
                                     file_types=[".xlsx", ".xls", ".csv"], type="filepath")
                bench_select = gr.Radio(
                    choices=["Compare with World Top 20", "Compare with India Top 20",
                             "Compare with All (CU + India + World)"],
                    value="Compare with All (CU + India + World)",
                    label="Compare uploaded dataset against")
            analyze_btn = gr.Button("🚀 Analyse & compare", variant="primary")
            eda_summary_md = gr.Markdown()
            with gr.Row():
                live_plot1 = gr.Plot(label="Research area benchmark comparison")
                live_plot2 = gr.Plot(label="Uploaded dataset SDG profile")
            live_table = gr.Dataframe(label="Uploaded dataset comparison table", interactive=False)
            download_excel = gr.File(label="📥 Download complete Excel analysis report")
            analyze_btn.click(analyze_custom_uploaded_file,
                inputs=[file_input, bench_select],
                outputs=[eda_summary_md, live_plot1, live_plot2, live_table, download_excel])

    gr.Markdown(
        "<center><small>Research portfolio shares are calculated over the 20 mapped research areas. "
        "AI title mapping and SDG alignment are analytical estimates, not official rankings.</small></center>"
    )

demo.launch(share=True, debug=False)
